
# T3 — Optimized pre-registered robustness, correction generalization, family sensitivity, and kinetic bridge

This notebook preserves the **same scientific analysis and pre-registered decision rules as T2**, but changes the computational architecture to minimize repeated work.

## Computational strategy

### One-time caches
For each `material × branch × temporal_mode`, the notebook computes the exact supported transition set **once** and caches:

- transition metadata;
- `q_full0`, `q_full1`;
- `q_single0`, `q_single1`;
- climate integer codes;
- climate-level \(H_{\rm full}\), \(H_{\rm single}\), and temperature-exposure summaries.

### Vectorized pair calculations
Climate-pair quantities are built with `np.triu_indices`, not Python `itertools.combinations`.

### LDF optimization
For each `material × branch`, hourly supported segments are prepared once. The notebook caches:

- hourly target arrays for full and single representations;
- temperatures and segment boundaries;
- equilibrium full-surface \(H\) by climate.

Only the LDF recurrence is repeated across \(\tau_k\) and \(E_a\). If `numba` is available, the recurrence is JIT-compiled; otherwise an equivalent NumPy/Python fallback is used.

### Bootstrap optimization
Climate-level sufficient statistics are cached first. Bootstrap replicates resample climate rows and use vectorized pair construction. No equilibrium interpolation or support reconstruction occurs inside the bootstrap loop.

---

## Pre-registered tests

### A. MgAl–PEI interpolation sensitivity
- Five temperature representations × 15 temporal phases.
- Directional robustness criterion: **all 5 × 15 results have \(C<0\)**.
- Primary outputs: \(C\) magnitude range and reversal count.
- `Q_LinearND` versus endpoint-linear is reported explicitly.

### B. Generalization of \(C\)
All mechanism classes A/B/C are included.

Primary pair-normalized quantity:
\[
C_{\rm rel}=\frac{|C|}{\bar H_{\rm single}}.
\]

\[
\frac{|C|}{|\Delta H_{\rm single}|}
\]
is retained only as a reversal diagnostic.

Prospective analysis:
\[
|C|/\bar H
\quad\text{vs}\quad
\left|
\langle |T-T_{\rm ref}|\rangle_A-
\langle |T-T_{\rm ref}|\rangle_B
\right|.
\]

### C. Family-level uncertainty
Report:
1. family-level results;
2. leave-one-family-out;
3. within-family climate bootstrap.

### D. Kinetic bridge
- Native 1 h weather.
- spin-up excluded.
- sanity: \(\tau_k=10^{-3}\) h, \(E_a=0\), Kendall \(\tau\ge0.999\).
- bridge: constant-\(k\), \(0.25\le\tau_k\le1\) h, Kendall \(\tau\ge0.9\).
- full-LDF vs equilibrium \(H\);
- single-LDF vs full-LDF;
- Arrhenius \(E_a=0,25,50\) kJ mol\(^{-1}\).

No LDF output is called productivity.


In [ ]:

# Colab setup and paths
from google.colab import drive, files as colab_files
drive.mount('/content/drive')

from pathlib import Path
from itertools import combinations
import warnings, math, json, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import LinearNDInterpolator, interp1d
from scipy.spatial import Delaunay
from scipy.stats import kendalltau, spearmanr

EPS = 1e-12
RNG = np.random.default_rng(20261013)

DRIVE = Path('/content/drive/MyDrive')
preferred = DRIVE/'climate_screening_v1_3_1_uncertainty_audit'/'climate_screening_backbone'
if preferred.exists():
    PROJECT = preferred
else:
    hits = list(DRIVE.rglob('climate_screening_backbone'))
    if not hits:
        raise FileNotFoundError('climate_screening_backbone not found on Drive.')
    PROJECT = hits[0]

QDIR = PROJECT/'results/Q_confirmatory_nested_uncertainty_timescale'
S2DIR = PROJECT/'results/S2_trajectory_null_gate_standalone'
OUT = PROJECT/'results/T_submission_robustness_process_link'
FIG = PROJECT/'figures/T_submission_robustness_process_link'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

CLIMATE = PROJECT/'data/processed/climate_hourly_2015_2024.csv'
PEI = PROJECT/'data/raw/PEI_climate_measured_capacity.csv'

def first_existing(patterns, roots=(DRIVE,)):
    for root in roots:
        for pat in patterns:
            hits = list(root.rglob(pat))
            if hits:
                return hits[0]
    return None

MASTER = first_existing([
    'q_RH_T_master_cycle08_audited_20261010(1).xlsx',
    'q_RH_T_master_cycle08_audited_20261010.xlsx',
    'q_RH_T_master_combined_v2(1).xlsx',
    'q_RH_T_master_combined_v2.xlsx',
])

if MASTER is None:
    print('Upload the equilibrium master workbook used for Q.')
    up = colab_files.upload()
    xls = [n for n in up if n.lower().endswith(('.xlsx','.xls'))]
    if not xls:
        raise FileNotFoundError('Equilibrium master workbook is required.')
    MASTER = Path('/content')/xls[0]

def resolve_or_upload(candidates, label):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    print(f'Upload {label}')
    up = colab_files.upload()
    for n in up:
        if any(Path(n).name == Path(c).name for c in candidates):
            return Path('/content')/n
    if len(up) == 1:
        return Path('/content')/next(iter(up))
    raise FileNotFoundError(label)

NESTED_INCREMENTAL = resolve_or_upload(
    [QDIR/'nested_incremental_value.csv'],
    'nested_incremental_value.csv'
)

S2_GATE = resolve_or_upload(
    [
        S2DIR/'S2_gate_by_material.csv',
        PROJECT/'results/S2_trajectory_null_gate_standalone'/'S2_gate_by_material.csv',
    ],
    'S2_gate_by_material.csv'
)

S2_EQUIV = resolve_or_upload(
    [
        S2DIR/'S2_null_equivalence_diagnostic_all'/'S2_null_equivalence_all_groups.csv',
        PROJECT/'results/S2_trajectory_null_gate_standalone'/'S2_null_equivalence_all_groups.csv',
    ],
    'S2_null_equivalence_all_groups.csv'
)

print('PROJECT:', PROJECT)
print('MASTER :', MASTER)
print('OUT    :', OUT)


In [ ]:

# Lightweight timing helper
import time
_TIMING = {}

class Timer:
    def __init__(self,name):
        self.name=name
    def __enter__(self):
        self.t0=time.perf_counter()
        return self
    def __exit__(self,*args):
        dt=time.perf_counter()-self.t0
        _TIMING[self.name]=dt
        print(f'[{self.name}] {dt:.2f} s')

def timing_report():
    if not _TIMING:
        return pd.DataFrame(columns=['stage','seconds'])
    return pd.DataFrame(
        [{'stage':k,'seconds':v} for k,v in _TIMING.items()]
    ).sort_values('seconds',ascending=False)


## 1. Rebuild the equilibrium representations and exact Q support

In [ ]:

def normalize_columns(df):
    d = df.copy()
    ren = {}
    if 'T_C' in d.columns: ren['T_C'] = 'temperature_C'
    if 'RH_pct' in d.columns: ren['RH_pct'] = 'rh_pct'
    if 'RH' in d.columns and 'rh_pct' not in d.columns: ren['RH'] = 'rh_pct'
    if 'q' in d.columns and 'q_mmol_g' not in d.columns: ren['q'] = 'q_mmol_g'
    if 'q_mmol_g_mean' in d.columns and 'q_mmol_g' not in d.columns:
        ren['q_mmol_g_mean'] = 'q_mmol_g'
    d = d.rename(columns=ren)
    if 'temperature_C' not in d.columns and 'temperature_K' in d.columns:
        d['temperature_C'] = d.temperature_K - 273.15
    return d

UNCERTAINTY_CANDIDATES = [
    'q_sd','q_std','q_sigma','q_uncertainty',
    'uncertainty_mmol_g','sd_mmol_g','std_mmol_g',
    'q_error','q_sem'
]

def clean(df):
    d = normalize_columns(df)
    need = ['temperature_C','rh_pct','q_mmol_g']
    if not set(need).issubset(d.columns):
        return pd.DataFrame(columns=need+['q_sigma'])
    sigcol = next((c for c in UNCERTAINTY_CANDIDATES if c in d.columns), None)
    cols = need + ([sigcol] if sigcol else [])
    d = d[cols].replace([np.inf,-np.inf],np.nan).dropna(subset=need)
    agg = {'q_mmol_g':'mean'}
    if sigcol: agg[sigcol] = 'mean'
    d = d.groupby(['temperature_C','rh_pct'],as_index=False).agg(agg)
    d = d.rename(columns={sigcol:'q_sigma'}) if sigcol else d.assign(q_sigma=np.nan)
    return d

climate = pd.read_csv(
    CLIMATE,
    usecols=['climate_class','city','timestamp_utc','temperature_C','RH_pct']
)
climate['timestamp_utc'] = pd.to_datetime(climate.timestamp_utc, utc=True, errors='coerce')
climate = climate.dropna().sort_values(
    ['climate_class','city','timestamp_utc']
).reset_index(drop=True)

materials = {}

if PEI.exists():
    d = pd.read_csv(PEI)
    if 'material' in d.columns:
        for m,g in d.groupby('material'):
            z = clean(g)
            if len(z):
                materials[str(m)] = z

raw = pd.read_excel(MASTER, sheet_name='Raw_Data')
mcol = 'Material' if 'Material' in raw.columns else 'material'
for m,g in raw.groupby(mcol):
    z = clean(g)
    if len(z) and (str(m) not in materials or len(z) > len(materials[str(m)])):
        materials[str(m)] = z

for pth in [
    PROJECT/'data/processed/validation_IRA_RHT.csv',
    PROJECT/'data/raw/validation_IRA_RHT.csv',
    PROJECT/'data/raw/IRA_A500_RHT.csv',
]:
    if pth.exists():
        d = pd.read_csv(pth)
        if 'material' in d.columns:
            for m,g in d.groupby('material'):
                z = clean(g)
                if len(z) and (str(m) not in materials or len(z) > len(materials[str(m)])):
                    materials[str(m)] = z

rows = []
for m,d in materials.items():
    temps = sorted(d.temperature_C.unique())
    if len(temps) < 3:
        continue
    Tlow, Thigh = temps[0], temps[-1]
    Tref = min(temps, key=lambda x: abs(x - 0.5*(Tlow+Thigh)))
    if Tref in [Tlow, Thigh]:
        continue
    nlow = len(d[d.temperature_C == Tlow])
    nref = len(d[d.temperature_C == Tref])
    nhigh = len(d[d.temperature_C == Thigh])
    if min(nlow,nref,nhigh) < 2:
        continue
    rows.append(dict(
        material=m, Tlow=Tlow, Tref=Tref, Thigh=Thigh,
        n_low=nlow, n_ref=nref, n_high=nhigh
    ))

elig = pd.DataFrame(rows)
display(elig)


In [ ]:

class FullSurface:
    def __init__(self,d):
        pts = d[['rh_pct','temperature_C']].to_numpy(float)
        vals = d.q_mmol_g.to_numpy(float)
        self.tri = Delaunay(pts)
        self.fun = LinearNDInterpolator(self.tri, vals, fill_value=np.nan)
    def eval(self,rh,T):
        return np.asarray(self.fun(np.asarray(rh,float), np.asarray(T,float)), float)

class SingleIso:
    def __init__(self,d,Tref):
        z = d[d.temperature_C == Tref].sort_values('rh_pct')
        self.Tref = float(Tref)
        self.rmin = float(z.rh_pct.min())
        self.rmax = float(z.rh_pct.max())
        self.fun = interp1d(
            z.rh_pct,z.q_mmol_g,kind='linear',
            bounds_error=False,fill_value=np.nan
        )
    def eval(self,rh,T):
        return np.asarray(self.fun(np.asarray(rh,float)),float)

class PairSlice:
    def __init__(self,d,T1,T2):
        self.T1 = float(min(T1,T2))
        self.T2 = float(max(T1,T2))
        z1 = d[d.temperature_C == self.T1].sort_values('rh_pct')
        z2 = d[d.temperature_C == self.T2].sort_values('rh_pct')
        self.rmin = max(float(z1.rh_pct.min()), float(z2.rh_pct.min()))
        self.rmax = min(float(z1.rh_pct.max()), float(z2.rh_pct.max()))
        self.f1 = interp1d(z1.rh_pct,z1.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
        self.f2 = interp1d(z2.rh_pct,z2.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
    def eval(self,rh,T):
        rh = np.asarray(rh,float)
        T = np.asarray(T,float)
        q1 = np.asarray(self.f1(rh),float)
        q2 = np.asarray(self.f2(rh),float)
        lam = (T-self.T1)/(self.T2-self.T1)
        out = (1-lam)*q1 + lam*q2
        out[(T<self.T1)|(T>self.T2)] = np.nan
        return out

class ThreeSlice:
    def __init__(self,d,Tlow,Tref,Thigh):
        self.Tlow, self.Tref, self.Thigh = map(float,(Tlow,Tref,Thigh))
        zl = d[d.temperature_C == Tlow].sort_values('rh_pct')
        zr = d[d.temperature_C == Tref].sort_values('rh_pct')
        zh = d[d.temperature_C == Thigh].sort_values('rh_pct')
        self.rmin = max(float(zl.rh_pct.min()),float(zr.rh_pct.min()),float(zh.rh_pct.min()))
        self.rmax = min(float(zl.rh_pct.max()),float(zr.rh_pct.max()),float(zh.rh_pct.max()))
        self.fl = interp1d(zl.rh_pct,zl.q_mmol_g,bounds_error=False,fill_value=np.nan)
        self.fr = interp1d(zr.rh_pct,zr.q_mmol_g,bounds_error=False,fill_value=np.nan)
        self.fh = interp1d(zh.rh_pct,zh.q_mmol_g,bounds_error=False,fill_value=np.nan)
    def eval(self,rh,T):
        rh = np.asarray(rh,float); T = np.asarray(T,float)
        ql = np.asarray(self.fl(rh),float)
        qr = np.asarray(self.fr(rh),float)
        qh = np.asarray(self.fh(rh),float)
        out = np.full(np.broadcast(rh,T).shape,np.nan,float)
        lo = (T>=self.Tlow)&(T<=self.Tref)
        hi = (T>=self.Tref)&(T<=self.Thigh)
        if np.any(lo):
            lam = (T[lo]-self.Tlow)/(self.Tref-self.Tlow)
            out[lo] = (1-lam)*ql[lo] + lam*qr[lo]
        if np.any(hi):
            lam = (T[hi]-self.Tref)/(self.Thigh-self.Tref)
            out[hi] = (1-lam)*qr[hi] + lam*qh[hi]
        return out

model = {}
for _,r in elig.iterrows():
    d = materials[r.material]
    model[r.material] = dict(
        full=FullSurface(d),
        single_ref=SingleIso(d,r.Tref),
        nested_low=PairSlice(d,r.Tlow,r.Tref),
        nested_high=PairSlice(d,r.Tref,r.Thigh),
        three_slice=ThreeSlice(d,r.Tlow,r.Tref,r.Thigh),
    )


In [ ]:

def subsample_climate(df, step, offset=0):
    out=[]
    for (cc,city),g in df.groupby(['climate_class','city'],sort=False):
        z=g.sort_values('timestamp_utc').iloc[offset::step].copy()
        if len(z):
            out.append(z)
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()

def average_climate(df, hours, offset=0):
    out=[]
    shift=pd.Timedelta(hours=int(offset))
    for (cc,city),g in df.groupby(['climate_class','city'],sort=False):
        z=g[['timestamp_utc','temperature_C','RH_pct']].copy()
        z['timestamp_shifted']=z.timestamp_utc-shift
        z=(z.set_index('timestamp_shifted')[['temperature_C','RH_pct']]
             .resample(f'{hours}h').mean().dropna().reset_index()
             .rename(columns={'timestamp_shifted':'timestamp_utc'}))
        z['timestamp_utc']=z.timestamp_utc+shift
        z['climate_class']=cc
        z['city']=city
        out.append(z)
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()

BASE_MODES = {
    'subsample_1h': climate.copy(),
    'subsample_3h': subsample_climate(climate,3,0),
    'subsample_6h': subsample_climate(climate,6,0),
    'subsample_24h': subsample_climate(climate,24,0),
    'average_6h': average_climate(climate,6,0),
    'average_24h': average_climate(climate,24,0),
}

def base_transitions(cdf,Tlo,Thi,Rlo,Rhi):
    frames=[]
    for (cc,city),g in cdf.groupby(['climate_class','city'],sort=False):
        g=g.sort_values('timestamp_utc')
        rh=g.RH_pct.to_numpy(float)
        T=g.temperature_C.to_numpy(float)
        ts=g.timestamp_utc.to_numpy()
        ok=(
            (rh[:-1]>=Rlo)&(rh[:-1]<=Rhi)&
            (rh[1:]>=Rlo)&(rh[1:]<=Rhi)&
            (T[:-1]>=Tlo)&(T[:-1]<=Thi)&
            (T[1:]>=Tlo)&(T[1:]<=Thi)
        )
        ix=np.flatnonzero(ok)
        if ix.size:
            frames.append(pd.DataFrame({
                'climate_class':cc,'city':city,
                'seq_idx':ix.astype(np.int32),
                'RH0':rh[ix],'RH1':rh[ix+1],
                'T0':T[ix],'T1':T[ix+1],
                'timestamp_utc':pd.to_datetime(ts[ix],utc=True),
            }))
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

def exact_q_common_mask(base,md,branch):
    reps = ['full','single_ref','nested_low','three_slice'] if branch=='lower' else \
           ['full','single_ref','nested_high','three_slice']
    mask=np.ones(len(base),dtype=bool)
    rh0=base.RH0.to_numpy(float); rh1=base.RH1.to_numpy(float)
    T0=base.T0.to_numpy(float); T1=base.T1.to_numpy(float)
    for rep in reps:
        obj=md[rep]
        q0=obj.eval(rh0,T0); q1=obj.eval(rh1,T1)
        mask &= np.isfinite(q0)&np.isfinite(q1)
    return mask

def exact_q_R_support(md,branch):
    reps = ['full','single_ref','nested_low','three_slice'] if branch=='lower' else \
           ['full','single_ref','nested_high','three_slice']
    rmins=[]; rmaxs=[]
    for rep in reps:
        obj=md[rep]
        if hasattr(obj,'rmin'):
            rmins.append(float(obj.rmin)); rmaxs.append(float(obj.rmax))
    return max([0.0]+rmins),min([100.0]+rmaxs)

def branch_bounds(er,branch):
    if branch=='lower': return float(er.Tlow),float(er.Tref)
    if branch=='upper': return float(er.Tref),float(er.Thigh)
    raise ValueError(branch)

# -----------------------------
# Fast grouping utilities
# -----------------------------
def climate_codes_from_df(df):
    key=(df.climate_class.astype(str)+'||'+df.city.astype(str)).to_numpy()
    labels, codes=np.unique(key,return_inverse=True)
    return labels,codes.astype(np.int32)

def bincount_sum(codes,values,n):
    return np.bincount(codes,weights=np.asarray(values,float),minlength=n)

def bincount_count(codes,n):
    return np.bincount(codes,minlength=n).astype(np.int64)

# -----------------------------
# Core cache
# -----------------------------
PROBLEM_CACHE={}

def build_problem_cache(material,branch,mode):
    key=(material,branch,mode)
    if key in PROBLEM_CACHE:
        return PROBLEM_CACHE[key]

    er=elig.set_index('material').loc[material]
    Tlo,Thi=branch_bounds(er,branch)
    md=model[material]
    Rlo,Rhi=exact_q_R_support(md,branch)

    base=base_transitions(BASE_MODES[mode],Tlo,Thi,Rlo,Rhi)
    if base.empty:
        PROBLEM_CACHE[key]=None
        return None

    base=base.loc[exact_q_common_mask(base,md,branch)].reset_index(drop=True)
    if base.empty:
        PROBLEM_CACHE[key]=None
        return None

    rh0=base.RH0.to_numpy(float); rh1=base.RH1.to_numpy(float)
    T0=base.T0.to_numpy(float); T1=base.T1.to_numpy(float)

    qf0=md['full'].eval(rh0,T0); qf1=md['full'].eval(rh1,T1)
    qs0=md['single_ref'].eval(rh0,T0); qs1=md['single_ref'].eval(rh1,T1)

    labels,codes=climate_codes_from_df(base)
    ncl=len(labels)
    N=bincount_count(codes,ncl)
    Hf=0.5*bincount_sum(codes,np.abs(qf1-qf0),ncl)
    Hs=0.5*bincount_sum(codes,np.abs(qs1-qs0),ncl)
    Tdev_step=0.5*(np.abs(T0-er.Tref)+np.abs(T1-er.Tref))
    Tdev_num=bincount_sum(codes,Tdev_step,ncl)
    Tdev=Tdev_num/np.maximum(N,1)

    cc=[]; city=[]
    for s in labels:
        a,b=s.split('||',1)
        cc.append(a); city.append(b)

    climate_table=pd.DataFrame({
        'climate_key':labels,
        'climate_class':cc,'city':city,
        'N':N,'H_full':Hf,'H_single':Hs,
        'Tdev_occ':Tdev,
    })

    cache={
        'material':material,'branch':branch,'mode':mode,
        'Tref':float(er.Tref),'Tlo':Tlo,'Thi':Thi,
        'Rlo':Rlo,'Rhi':Rhi,
        'base':base,
        'RH0':rh0,'RH1':rh1,'T0':T0,'T1':T1,
        'qf0':qf0,'qf1':qf1,'qs0':qs0,'qs1':qs1,
        'labels':labels,'codes':codes,
        'climate_table':climate_table,
    }
    PROBLEM_CACHE[key]=cache
    return cache

def exact_problem(material,branch,mode_or_df):
    # Backward-compatible wrapper.
    if isinstance(mode_or_df,str):
        c=build_problem_cache(material,branch,mode_or_df)
        return pd.DataFrame() if c is None else c['base'].copy()

    # Nonstandard phase data are used only by Mg interpolation sensitivity.
    er=elig.set_index('material').loc[material]
    Tlo,Thi=branch_bounds(er,branch)
    md=model[material]
    Rlo,Rhi=exact_q_R_support(md,branch)
    base=base_transitions(mode_or_df,Tlo,Thi,Rlo,Rhi)
    if base.empty:
        return base
    return base.loc[exact_q_common_mask(base,md,branch)].reset_index(drop=True)

def score_H_cached(material,branch,mode,which='full'):
    c=build_problem_cache(material,branch,mode)
    if c is None: return pd.DataFrame()
    col='H_full' if which=='full' else 'H_single'
    out=c['climate_table'][['climate_class','city','N',col]].copy()
    return out.rename(columns={col:'H'})

print('Core reconstruction ready with caching.')



## 2. Test A — MgAl–PEI \(C\): interpolation magnitude and reversal sensitivity

For a monotone endpoint interpolation, the pointwise sign of \(q_{\rm interp}-q_{\rm single}\) is often structurally constrained by the endpoint difference. Therefore **sign retention alone is not the main result**.

The primary questions are:

1. How wide is the **magnitude range of \(C\)** across interpolation forms?
2. How often does a climate-pair **rank reversal** appear or disappear?
3. Does `Q_LinearND` differ from direct endpoint-linear interpolation?

The endpoint-sqrt and endpoint-square warps are deliberately opposite concave/convex distortions of the normalized temperature coordinate and serve as simple bounding stresses.


In [ ]:

class EndpointWarpSurface:
    def __init__(self,d,T1,T2,warp='linear'):
        self.T1 = float(min(T1,T2)); self.T2 = float(max(T1,T2))
        z1 = d[d.temperature_C==self.T1].sort_values('rh_pct')
        z2 = d[d.temperature_C==self.T2].sort_values('rh_pct')
        self.rmin = max(float(z1.rh_pct.min()),float(z2.rh_pct.min()))
        self.rmax = min(float(z1.rh_pct.max()),float(z2.rh_pct.max()))
        self.f1 = interp1d(z1.rh_pct,z1.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
        self.f2 = interp1d(z2.rh_pct,z2.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
        self.warp = warp
    def weight(self,lam):
        lam = np.asarray(lam,float)
        if self.warp=='linear':
            return lam
        if self.warp=='sqrt':
            return np.sqrt(np.clip(lam,0,1))
        if self.warp=='square':
            return np.clip(lam,0,1)**2
        if self.warp=='smoothstep':
            x = np.clip(lam,0,1)
            return 3*x*x - 2*x*x*x
        raise ValueError(self.warp)
    def eval(self,rh,T):
        rh=np.asarray(rh,float); T=np.asarray(T,float)
        q1=np.asarray(self.f1(rh),float); q2=np.asarray(self.f2(rh),float)
        lam=(T-self.T1)/(self.T2-self.T1)
        w=self.weight(lam)
        out=(1-w)*q1+w*q2
        out[(T<self.T1)|(T>self.T2)] = np.nan
        return out

MG = 'Mg0.55Al-CO3_PEI(30)'
if MG not in materials:
    raise KeyError(f'{MG} not found in loaded materials.')

er = elig.set_index('material').loc[MG]
mg_schemes = {
    'Q_LinearND': model[MG]['full'],
    'endpoint_linear': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'linear'),
    'endpoint_sqrt': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'sqrt'),
    'endpoint_square': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'square'),
    'endpoint_smoothstep': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'smoothstep'),
}


In [ ]:

A_CC,A_CITY = 'monsoonal_continental','beijing'
B_CC,B_CITY = 'monsoonal_continental','shenyang'

phase_specs=[]
for off in range(3):
    phase_specs.append(('subsample_3h',off,subsample_climate(climate,3,off)))
for off in range(6):
    phase_specs.append(('subsample_6h',off,subsample_climate(climate,6,off)))
for off in range(6):
    phase_specs.append(('average_6h',off,average_climate(climate,6,off)))

def pair_delta_fast(labels,H,keyA=f'{A_CC}||{A_CITY}',keyB=f'{B_CC}||{B_CITY}'):
    pos={k:i for i,k in enumerate(labels)}
    if keyA not in pos or keyB not in pos: return np.nan
    return float(H[pos[keyA]]-H[pos[keyB]])

# Precompute each phase once.
MG_PHASE_CACHE={}
for family,off,cdf in phase_specs:
    z=exact_problem(MG,'lower',cdf)
    if z.empty:
        continue

    rh0=z.RH0.to_numpy(float); rh1=z.RH1.to_numpy(float)
    T0=z.T0.to_numpy(float); T1=z.T1.to_numpy(float)
    labels,codes=climate_codes_from_df(z)
    ncl=len(labels)

    qs0=model[MG]['single_ref'].eval(rh0,T0)
    qs1=model[MG]['single_ref'].eval(rh1,T1)
    Hs=0.5*bincount_sum(codes,np.abs(qs1-qs0),ncl)

    MG_PHASE_CACHE[(family,off)]={
        'z':z,'rh0':rh0,'rh1':rh1,'T0':T0,'T1':T1,
        'labels':labels,'codes':codes,'ncl':ncl,
        'qs0':qs0,'qs1':qs1,'Hs':Hs,
        'Tmid':0.5*(T0+T1),'RHmid':0.5*(rh0+rh1),
    }

mg_rows=[]
for (family,off),pc in MG_PHASE_CACHE.items():
    labels=pc['labels']; codes=pc['codes']; ncl=pc['ncl']
    d_single=pair_delta_fast(labels,pc['Hs'])

    for scheme,obj in mg_schemes.items():
        qa0=obj.eval(pc['rh0'],pc['T0'])
        qa1=obj.eval(pc['rh1'],pc['T1'])
        Halt=0.5*bincount_sum(codes,np.abs(qa1-qa0),ncl)
        d_alt=pair_delta_fast(labels,Halt)
        C=d_alt-d_single

        corr=0.5*(np.abs(qa1-qa0)-np.abs(pc['qs1']-pc['qs0']))
        region=(pc['Tmid'] < -10)&(pc['RHmid']>=40)&(pc['RHmid']<=70)

        keyA=f'{A_CC}||{A_CITY}'; keyB=f'{B_CC}||{B_CITY}'
        pos={k:i for i,k in enumerate(labels)}
        if keyA in pos and keyB in pos:
            cA=bincount_sum(codes, corr*region, ncl)[pos[keyA]]
            cB=bincount_sum(codes, corr*region, ncl)[pos[keyB]]
            C_region=float(cA-cB)
        else:
            C_region=np.nan

        mg_rows.append(dict(
            phase_family=family,offset_h=off,scheme=scheme,
            delta_Hsingle=d_single,delta_Halt=d_alt,C=C,
            sign_C=np.sign(C),abs_C=abs(C),
            region_C=C_region,
            region_fraction_signed=(C_region/C if abs(C)>EPS else np.nan),
            reversal=(
                np.sign(d_single)!=0 and np.sign(d_alt)!=0 and
                np.sign(d_single)!=np.sign(d_alt)
            ),
        ))

mg_interp=pd.DataFrame(mg_rows)
mg_interp.to_csv(OUT/'Mg_C_interpolation_sensitivity_by_phase.csv',index=False)

mg_interp_summary=(
    mg_interp.groupby('scheme',as_index=False)
    .agg(
        n_phases=('C','size'),
        n_C_negative=('C',lambda x:int(np.sum(np.asarray(x)<0))),
        frac_C_negative=('C',lambda x:float(np.mean(np.asarray(x)<0))),
        C_min=('C','min'),C_median=('C','median'),C_max=('C','max'),
        absC_median=('abs_C','median'),
        reversal_count=('reversal','sum'),
        reversal_fraction=('reversal','mean'),
        region_fraction_median=('region_fraction_signed','median'),
    )
)
mg_interp_summary['C_span']=mg_interp_summary.C_max-mg_interp_summary.C_min
mg_interp_summary.to_csv(OUT/'Mg_C_interpolation_sensitivity_summary.csv',index=False)
display(mg_interp_summary)

wide=mg_interp.pivot_table(
    index=['phase_family','offset_h'],columns='scheme',
    values=['C','reversal'],aggfunc='first'
)
if ('C','Q_LinearND') in wide.columns and ('C','endpoint_linear') in wide.columns:
    qlin=pd.DataFrame({
        'phase_family':wide.index.get_level_values(0),
        'offset_h':wide.index.get_level_values(1),
        'C_Q_LinearND':wide[('C','Q_LinearND')].to_numpy(float),
        'C_endpoint_linear':wide[('C','endpoint_linear')].to_numpy(float),
    })
    qlin['delta_C_QminusEndpointLinear']=qlin.C_Q_LinearND-qlin.C_endpoint_linear
    qlin['abs_delta_C']=np.abs(qlin.delta_C_QminusEndpointLinear)
    qlin.to_csv(OUT/'Mg_QLinearND_vs_endpointLinear.csv',index=False)
    display(qlin)

fig,ax=plt.subplots(figsize=(8.5,4.8))
phase_order=mg_interp[['phase_family','offset_h']].drop_duplicates().reset_index(drop=True)
phase_order['phase_id']=np.arange(len(phase_order))
plotd=mg_interp.merge(phase_order,on=['phase_family','offset_h'])
for scheme,g in plotd.groupby('scheme'):
    ax.plot(g.phase_id,g.C,marker='o',label=scheme)
ax.axhline(0,linewidth=1)
ax.set_ylabel('C = ΔH_alt - ΔH_single')
ax.set_xlabel('Temporal phase')
ax.set_title('Mg lower: interpolation sensitivity of correction magnitude')
ax.legend(fontsize=8,ncol=2)
fig.tight_layout()
fig.savefig(FIG/'Mg_C_interpolation_sensitivity.png',dpi=220)
plt.show()

direction_pass=bool((mg_interp_summary.frac_C_negative==1).all())
print('Directional robustness criterion (all 5×15 C<0):',
      'PASS' if direction_pass else 'FAIL')



### Interpretation rule for Test A

- **Strong mechanism claim:** all plausible endpoint-preserving interpolation schemes retain the same sign of \(C\); only magnitude changes.
- **Moderate claim:** most schemes retain the sign, but one plausible scheme crosses zero.
- **Weak claim:** sign depends materially on interpolation choice.

The statement should remain endpoint-based: the measured low-temperature and reference isotherms differ; the intermediate-temperature magnitude is model-dependent.



## 3. Test B — Generalize \(C\) across all mechanism classes

The correction
\[
C=\Delta H_{\rm full}-\Delta H_{\rm single}
\]
does not require nonlinear RH shape. Therefore this analysis includes **class A, class B, and class C**.

Two distinctions are kept explicit:

- **held-interior cells**: temperature information can be checked against an independently measured interior temperature;
- **non-held-interior cells**: the fuller branch remains interpolation-constrained.

Primary pair scale:
\[
C_{\rm rel}=\frac{|C|}{\bar H_{\rm single}}.
\]

The ratio
\[
\frac{|C|}{|\Delta H_{\rm single}|}
\]
is retained only to diagnose whether the correction is large enough to overcome the existing ranking margin.


In [ ]:

diag = pd.read_csv(S2_EQUIV)
gate = pd.read_csv(S2_GATE)

if 'delta_rho_partial' not in gate.columns and 'delta_rho_partial_single_minus_null' in gate.columns:
    gate['delta_rho_partial'] = gate['delta_rho_partial_single_minus_null']
if 'strict_CI_pass' not in gate.columns and 'strict_gate_pass' in gate.columns:
    gate['strict_CI_pass'] = gate['strict_gate_pass']

# All mechanism classes are retained.
all_cells = diag[
    diag.mechanism_class.astype(str).str.startswith(('A_','B_','C_'))
].copy()

def n_internal_measured(material,branch):
    d=materials[str(material)]
    er=elig.set_index('material').loc[str(material)]
    lo,hi=branch_bounds(er,branch)
    temps=np.sort(d.temperature_C.unique().astype(float))
    return int(np.sum((temps>lo+1e-9)&(temps<hi-1e-9)))

all_cells['n_internal_measured']=[
    n_internal_measured(m,b) if m in materials and m in set(elig.material) else np.nan
    for m,b in zip(all_cells.material,all_cells.branch)
]
all_cells['held_interior']=all_cells.n_internal_measured.fillna(0).astype(int)>=1

print('Mechanism counts:')
display(all_cells.mechanism_class.value_counts(dropna=False).rename_axis('class').reset_index(name='n'))

print('\nHeld-interior cells:')
display(
    all_cells[all_cells.held_interior][
        ['material','branch','temporal_mode','mechanism_class','n_internal_measured']
    ].sort_values(['material','branch','temporal_mode'])
)


In [ ]:

def pair_decomposition_fast(material,branch,mode,mechanism_class=None,held_interior=False):
    c=build_problem_cache(material,branch,mode)
    if c is None:
        return pd.DataFrame(),pd.DataFrame()

    g=c['climate_table'].copy()
    n=len(g)
    if n<2:
        return g,pd.DataFrame()

    i,j=np.triu_indices(n,1)

    Hs=g.H_single.to_numpy(float)
    Hf=g.H_full.to_numpy(float)
    E=Hf-Hs
    Tv=g.Tdev_occ.to_numpy(float)

    ds=Hs[i]-Hs[j]
    df=Hf[i]-Hf[j]
    C=E[i]-E[j]
    Hbar=0.5*(np.abs(Hs[i])+np.abs(Hs[j]))
    Crel=np.abs(C)/(Hbar+EPS)
    margin_ratio=np.abs(C)/(np.abs(ds)+EPS)
    xT=np.abs(Tv[i]-Tv[j])
    margin=np.abs(ds)/(Hbar+EPS)

    reversal=(
        (np.sign(ds)!=0)&(np.sign(df)!=0)&
        (np.sign(ds)!=np.sign(df))
    )
    criterion=(
        (np.sign(C)!=0)&(np.sign(ds)!=0)&
        (np.sign(C)!=np.sign(ds))&
        (np.abs(C)>np.abs(ds))
    )

    pair=pd.DataFrame({
        'material':material,'branch':branch,'temporal_mode':mode,
        'mechanism_class':mechanism_class,
        'held_interior':bool(held_interior),
        'climate_A':g.climate_key.to_numpy()[i],
        'climate_B':g.climate_key.to_numpy()[j],
        'Hsingle_A':Hs[i],'Hsingle_B':Hs[j],
        'Hbar_single':Hbar,
        'delta_Hsingle':ds,'delta_Hfull':df,'C':C,
        'C_rel_Hbar':Crel,
        'absC_over_absSingle':margin_ratio,
        'Tdev_occ_A':Tv[i],'Tdev_occ_B':Tv[j],
        'delta_Tdev_occ':xT,
        'single_relative_margin':margin,
        'reversal':reversal,
        'criterion_predicts_reversal':criterion,
        'preserved_same_sign':~reversal,
    })
    return g,pair

climate_tables=[]
pair_tables=[]

# Build caches only for cells actually needed.
for _,r in all_cells.iterrows():
    m,b,mode=r.material,r.branch,r.temporal_mode
    if m not in model or mode not in BASE_MODES:
        continue
    gc,gp=pair_decomposition_fast(
        m,b,mode,r.mechanism_class,bool(r.held_interior)
    )
    if len(gc):
        gc=gc.copy()
        gc['material']=m;gc['branch']=b;gc['temporal_mode']=mode
        gc['mechanism_class']=r.mechanism_class
        gc['held_interior']=bool(r.held_interior)
        climate_tables.append(gc)
    if len(gp):
        pair_tables.append(gp)

G_climate=pd.concat(climate_tables,ignore_index=True) if climate_tables else pd.DataFrame()
G_pairs=pd.concat(pair_tables,ignore_index=True) if pair_tables else pd.DataFrame()

G_climate.to_csv(OUT/'C_generalization_allclasses_climate_level.csv',index=False)
G_pairs.to_csv(OUT/'C_generalization_allclasses_pair_level.csv',index=False)

if len(G_pairs):
    audit=G_pairs[
        (np.abs(G_pairs.delta_Hsingle)>1e-12)&
        (np.abs(G_pairs.delta_Hfull)>1e-12)
    ].copy()
    audit['criterion_matches_observed']=(
        audit.reversal==audit.criterion_predicts_reversal
    )
    print('Exact reversal-condition audit:',
          audit.criterion_matches_observed.mean(),
          f"({audit.criterion_matches_observed.sum()}/{len(audit)})")


In [ ]:

# Cell-level summaries. Pairwise observations are NOT used as independent inferential units.
if len(G_pairs):
    cell_rows=[]
    for key,g in G_pairs.groupby(
        ['material','branch','temporal_mode','mechanism_class','held_interior'],
        dropna=False
    ):
        m,b,mode,cls,held=key
        x=g.delta_Tdev_occ.to_numpy(float)
        y=g.C_rel_Hbar.to_numpy(float)
        ok=np.isfinite(x)&np.isfinite(y)

        rho=np.nan
        slope=np.nan
        intercept=np.nan
        if ok.sum()>=5 and np.nanstd(x[ok])>EPS:
            rho=float(spearmanr(x[ok],y[ok]).statistic)
            slope,intercept=np.polyfit(x[ok],y[ok],1)

        cell_rows.append(dict(
            material=m,branch=b,temporal_mode=mode,
            mechanism_class=cls,held_interior=held,
            n_pairs=len(g),
            median_C_rel_Hbar=float(np.nanmedian(y)),
            q90_C_rel_Hbar=float(np.nanquantile(y,.90)),
            reversal_fraction=float(np.mean(g.reversal)),
            median_absC_over_absSingle=float(np.nanmedian(g.absC_over_absSingle)),
            spearman_Crel_vs_deltaTdev=rho,
            slope_Crel_per_degC=float(slope) if np.isfinite(slope) else np.nan,
            intercept_Crel=float(intercept) if np.isfinite(intercept) else np.nan,
        ))

    C_cell_summary=pd.DataFrame(cell_rows)
    C_cell_summary.to_csv(OUT/'C_generalization_cell_summary.csv',index=False)
    display(C_cell_summary.sort_values(
        ['held_interior','mechanism_class','material','branch','temporal_mode'],
        ascending=[False,True,True,True,True]
    ))

    # Primary relationship figure: pair values shown faintly, cell-level slopes are the inferential objects.
    fig,ax=plt.subplots(figsize=(7,4.8))
    sample=G_pairs[np.isfinite(G_pairs.C_rel_Hbar)&np.isfinite(G_pairs.delta_Tdev_occ)].copy()
    ax.scatter(sample.delta_Tdev_occ,sample.C_rel_Hbar,s=8,alpha=.12)
    ax.set_xlabel(r'|Δ occupancy-weighted |T−Tref|| (°C)')
    ax.set_ylabel(r'|C| / mean H_single')
    ax.set_title('Correction magnitude versus differential temperature exposure')
    fig.tight_layout()
    fig.savefig(FIG/'Crel_vs_temperature_exposure_pairs.png',dpi=220)
    plt.show()

    print('\nHeld-interior cells (independent temperature information):')
    display(C_cell_summary[C_cell_summary.held_interior].sort_values(
        ['material','branch','temporal_mode']
    ))

    # Margin ratio is diagnostic only.
    ratio_bins=[0,.25,.5,1,2,5,np.inf]
    ratio_labels=['<0.25','0.25-0.5','0.5-1','1-2','2-5','>5']
    rr=G_pairs.copy()
    rr['margin_ratio_bin']=pd.cut(
        rr.absC_over_absSingle,ratio_bins,labels=ratio_labels,right=False
    )
    ratio_summary=(
        rr.groupby('margin_ratio_bin',observed=True,as_index=False)
          .agg(n_pairs=('reversal','size'),reversal_fraction=('reversal','mean'))
    )
    ratio_summary.to_csv(OUT/'C_margin_ratio_vs_reversal.csv',index=False)
    display(ratio_summary)



**Important:** the reversal condition is an identity once the fuller surface is known. Its scientific value is explanatory, not predictive. A practical stopping rule would require a pre-measurement bound or prior for \(|C|\).



## 4. Test C — Family-level summaries, LOFO, and within-family climate bootstrap

### Pre-specified family definition

**Primary definition: formulation lineage / host architecture.**

- PEI-30/40/50 wt% are one `PEI_loading_family`.
- Al₂O₃–TEPA and MIL-101(Cr)–TEPA remain separate primary families because the host architecture and formulation lineage differ.
- MgAl–PEI, IRA-900-C, ImCOF-TAEA, and LWT-bPPI5 are separate families.

**Sensitivity definition:** collapse Al₂O₃–TEPA and MIL-101(Cr)–TEPA into one `TEPA_supported_family`.

The primary mapping is fixed before examining the family-level results below.


In [ ]:

def chemistry_family_primary(m):
    m=str(m)
    if m in {'PEI-30wt','PEI-40wt','PEI-50wt'}:
        return 'PEI_loading_family'
    if 'IRA-900' in m:
        return 'IRA_900'
    if m.startswith('Mg0.55Al'):
        return 'MgAl_PEI'
    if 'Al2O3' in m:
        return 'Al2O3_TEPA'
    if 'MIL-101' in m:
        return 'MIL101_TEPA'
    if 'ImCOF' in m:
        return 'ImCOF_TAEA'
    if 'LWT' in m:
        return 'LWT_bPPI5'
    return m

def chemistry_family_sensitivity(m):
    p=chemistry_family_primary(m)
    if p in {'Al2O3_TEPA','MIL101_TEPA'}:
        return 'TEPA_supported_family'
    return p

if 'C_cell_summary' not in globals():
    raise RuntimeError('Run Test B first.')

C_cell_summary['family_primary']=C_cell_summary.material.map(chemistry_family_primary)
C_cell_summary['family_sensitivity']=C_cell_summary.material.map(chemistry_family_sensitivity)

# Family-level descriptive summaries
family_results=(
    C_cell_summary.groupby(
        ['family_primary','mechanism_class','held_interior'],
        as_index=False,dropna=False
    )
    .agg(
        n_cells=('material','size'),
        n_materials=('material','nunique'),
        median_C_rel_Hbar=('median_C_rel_Hbar','median'),
        median_reversal_fraction=('reversal_fraction','median'),
        median_exposure_slope=('slope_Crel_per_degC','median'),
        median_exposure_rho=('spearman_Crel_vs_deltaTdev','median'),
    )
)
family_results.to_csv(OUT/'family_level_C_results.csv',index=False)
display(family_results)

# Leave-one-family-out summaries
def lofo_summary(df,value_col,family_col='family_primary'):
    out=[]
    fams=sorted(df[family_col].dropna().unique())
    for omit in fams:
        z=df[df[family_col]!=omit]
        vals=(
            z.groupby(family_col,as_index=False)[value_col]
             .median()[value_col].dropna().to_numpy(float)
        )
        out.append(dict(
            omitted_family=omit,
            n_families_remaining=len(vals),
            family_median=float(np.median(vals)) if len(vals) else np.nan,
            family_min=float(np.min(vals)) if len(vals) else np.nan,
            family_max=float(np.max(vals)) if len(vals) else np.nan,
        ))
    return pd.DataFrame(out)

lofo_Crel=lofo_summary(C_cell_summary,'median_C_rel_Hbar')
lofo_slope=lofo_summary(C_cell_summary,'slope_Crel_per_degC')
lofo_Crel.to_csv(OUT/'LOFO_family_Crel.csv',index=False)
lofo_slope.to_csv(OUT/'LOFO_family_exposure_slope.csv',index=False)

print('LOFO: median normalized correction')
display(lofo_Crel)
print('LOFO: exposure slope')
display(lofo_slope)

# Sensitivity family definition
sens=(
    C_cell_summary.assign(
        family=C_cell_summary.material.map(chemistry_family_sensitivity)
    )
    .groupby('family',as_index=False)
    .agg(
        n_cells=('material','size'),
        median_C_rel_Hbar=('median_C_rel_Hbar','median'),
        median_exposure_slope=('slope_Crel_per_degC','median'),
    )
)
sens.to_csv(OUT/'family_definition_sensitivity.csv',index=False)
print('Sensitivity family mapping:')
display(sens)


In [ ]:

# Hierarchical bootstrap: within-family climate resampling.
# Optimized: no support reconstruction or interpolation inside bootstrap.

def bootstrap_pair_stats_from_climate_arrays(Hf,Hs,Tv,n_boot=1000,seed=20261013):
    rng=np.random.default_rng(seed)
    n=len(Hf)
    if n<5:
        return pd.DataFrame()

    rows=np.empty((n_boot,3),float)
    for b in range(n_boot):
        ix=rng.integers(0,n,n)
        hf=Hf[ix]; hs=Hs[ix]; tv=Tv[ix]

        ii,jj=np.triu_indices(n,1)
        C=(hf-hs)[ii]-(hf-hs)[jj]
        Hbar=0.5*(np.abs(hs[ii])+np.abs(hs[jj]))
        y=np.abs(C)/(Hbar+EPS)
        x=np.abs(tv[ii]-tv[jj])

        med=float(np.median(y))
        if np.std(x)>EPS:
            slope=float(np.polyfit(x,y,1)[0])
            rho=float(spearmanr(x,y).statistic)
        else:
            slope=np.nan; rho=np.nan
        rows[b]=(med,slope,rho)

    return pd.DataFrame(
        rows,
        columns=['median_C_rel_Hbar','slope_Crel_per_degC','spearman_Crel_vs_deltaTdev']
    ).assign(boot=np.arange(n_boot,dtype=int))

# Representative temporal mode per material/branch.
boot_targets=[]
for (m,b),g in C_cell_summary.groupby(['material','branch']):
    modes=list(g.temporal_mode)
    mode='subsample_3h' if 'subsample_3h' in modes else modes[0]
    boot_targets.append((m,b,mode))

boot_frames=[]
for k,(m,b,mode) in enumerate(boot_targets):
    c=build_problem_cache(m,b,mode)
    if c is None:
        continue
    gt=c['climate_table']
    bt=bootstrap_pair_stats_from_climate_arrays(
        gt.H_full.to_numpy(float),
        gt.H_single.to_numpy(float),
        gt.Tdev_occ.to_numpy(float),
        n_boot=1000,
        seed=20261013+k
    )
    if len(bt):
        bt['material']=m;bt['branch']=b;bt['temporal_mode']=mode
        bt['family_primary']=chemistry_family_primary(m)
        boot_frames.append(bt)

hier_boot=pd.concat(boot_frames,ignore_index=True) if boot_frames else pd.DataFrame()
hier_boot.to_csv(OUT/'hierarchical_within_family_climate_bootstrap.csv',index=False)

if len(hier_boot):
    hb_summary=(
        hier_boot.groupby('family_primary',as_index=False)
        .agg(
            n_boot=('boot','size'),
            Crel_median=('median_C_rel_Hbar','median'),
            Crel_lo=('median_C_rel_Hbar',lambda x:np.nanquantile(x,.025)),
            Crel_hi=('median_C_rel_Hbar',lambda x:np.nanquantile(x,.975)),
            slope_median=('slope_Crel_per_degC','median'),
            slope_lo=('slope_Crel_per_degC',
                      lambda x:np.nanquantile(x.dropna(),.025) if x.notna().any() else np.nan),
            slope_hi=('slope_Crel_per_degC',
                      lambda x:np.nanquantile(x.dropna(),.975) if x.notna().any() else np.nan),
        )
    )
    hb_summary.to_csv(OUT/'hierarchical_within_family_bootstrap_summary.csv',index=False)
    display(hb_summary)



With only three independent class-C chemistry groups, these intervals are **descriptive sensitivity intervals**, not strong asymptotic inference. The main role is to show how much formulation-level precision was inflated by counting the three PEI loadings separately.



## 5. Test D — Kinetic bridge: full-LDF, single-LDF, spin-up, and Arrhenius sensitivity

The weather series is integrated at the **native 1 h resolution**.

For either the fuller or single equilibrium representation:
\[
\frac{dq}{dt}=k(T)\,[q^*(RH,T)-q].
\]

Two kinetic models are evaluated:

1. **constant \(k\)**;
2. **Arrhenius \(k(T)\)**,
   \[
   k(T)=k_{\rm ref}
   \exp\left[
   -\frac{E_a}{R}
   \left(
   \frac{1}{T}-\frac{1}{T_{\rm ref}}
   \right)
   \right].
   \]

Pre-specified activation-energy sensitivities:
\[
E_a = 0,\ 25,\ 50\ {\rm kJ\,mol^{-1}}.
\]

Each contiguous supported segment is initialized at the first equilibrium state, and the first
\[
t_{\rm spin}=\max(6\,{\rm h},5\tau_k)
\]
is excluded from turnover accumulation.

Main comparisons:

- equilibrium \(H_{\rm full}\) vs `full-LDF`;
- `single-LDF` vs `full-LDF`;
- whether finite kinetics attenuates or amplifies the single/full ranking discrepancy;
- whether low-temperature Arrhenius slowing changes that conclusion.


In [ ]:

PROCESS_MATERIALS=[
    m for m in ['IRA-900-C','Mg0.55Al-CO3_PEI(30)']
    if m in model
]
PROCESS_BRANCHES=['lower','upper']
TAU_HOURS=[1e-3,0.05,0.25,0.5,1,3,6,12]
EA_KJMOL=[0,25,50]
PROCESS_MODE='subsample_1h'
R_GAS=8.314462618

# Optional JIT acceleration.
try:
    from numba import njit
    HAVE_NUMBA=True
except Exception:
    HAVE_NUMBA=False
    def njit(*args,**kwargs):
        def deco(f): return f
        return deco

print('Numba available:',HAVE_NUMBA)

# LDF cache: one preparation per material × branch.
LDF_CACHE={}

def prepare_ldf_cache(material,branch):
    key=(material,branch)
    if key in LDF_CACHE:
        return LDF_CACHE[key]

    c=build_problem_cache(material,branch,PROCESS_MODE)
    if c is None:
        LDF_CACHE[key]=None
        return None

    b=c['base'].copy()
    # Build contiguous segment ids without Python row loops.
    climate_key=(b.climate_class.astype(str)+'||'+b.city.astype(str)).to_numpy()
    seq=b.seq_idx.to_numpy(int)

    newseg=np.ones(len(b),dtype=bool)
    if len(b)>1:
        newseg[1:] = (climate_key[1:]!=climate_key[:-1]) | (seq[1:]!=seq[:-1]+1)
    seg_id=np.cumsum(newseg)-1

    # segment starts/ends
    starts=np.flatnonzero(newseg)
    ends=np.r_[starts[1:],len(b)]

    er=elig.set_index('material').loc[material]
    rh1=c['RH1']; T1=c['T1']
    qf_target=c['qf1']
    qs_target=c['qs1']

    # climate code per transition
    labels,codes=climate_codes_from_df(b)
    ncl=len(labels)

    # full equilibrium H on exact support, once
    H_eq_full=0.5*bincount_sum(codes,np.abs(c['qf1']-c['qf0']),ncl)

    out=dict(
        labels=labels,codes=codes,ncl=ncl,
        starts=starts.astype(np.int64),ends=ends.astype(np.int64),
        qf_target=np.asarray(qf_target,float),
        qs_target=np.asarray(qs_target,float),
        qf_init=np.asarray(c['qf0'],float),
        qs_init=np.asarray(c['qs0'],float),
        T1=np.asarray(T1,float),
        dt=1.0,
        Tref=float(er.Tref),
        H_eq_full=H_eq_full,
    )
    LDF_CACHE[key]=out
    return out

@njit(cache=True)
def _ldf_run_numba(target,init_q,T1,codes,starts,ends,ncl,
                   tau_h,Ea_kJmol,Tref_C,dt,Rgas):
    H=np.zeros(ncl,np.float64)
    Hbal=np.zeros(ncl,np.float64)
    Nkeep=np.zeros(ncl,np.int64)

    kref=1.0/max(tau_h,1e-12)
    spin_h=max(6.0,5.0*tau_h)
    TrefK=Tref_C+273.15
    Ea=Ea_kJmol*1000.0

    for s in range(len(starts)):
        a=starts[s]; b=ends[s]
        if b<=a:
            continue
        q=init_q[a]
        elapsed=0.0
        pos=0.0; neg=0.0
        cc=codes[a]

        for p in range(a,b):
            if Ea_kJmol==0.0:
                kT=kref
            else:
                TK=T1[p]+273.15
                kT=kref*np.exp(-(Ea/Rgas)*(1.0/TK-1.0/TrefK))
            alpha=1.0-np.exp(-kT*dt)
            qn=q+alpha*(target[p]-q)
            dq=qn-q
            elapsed += dt
            if elapsed>spin_h:
                H[cc]+=0.5*abs(dq)
                if dq>=0: pos+=dq
                else: neg+=-dq
                Nkeep[cc]+=1
            q=qn

        Hbal[cc]+=min(pos,neg)

    return H,Hbal,Nkeep

def run_ldf_cached(cache,tau_h,Ea_kJmol,which='full'):
    target=cache['qf_target'] if which=='full' else cache['qs_target']
    init_q=cache['qf_init'] if which=='full' else cache['qs_init']
    return _ldf_run_numba(
        np.asarray(target,float),np.asarray(init_q,float),
        np.asarray(cache['T1'],float),np.asarray(cache['codes'],np.int64),
        np.asarray(cache['starts'],np.int64),np.asarray(cache['ends'],np.int64),
        int(cache['ncl']),float(tau_h),float(Ea_kJmol),
        float(cache['Tref']),float(cache['dt']),float(R_GAS)
    )

process_rows=[]
process_detail_rows=[]

for m in PROCESS_MATERIALS:
    for branch in PROCESS_BRANCHES:
        lc=prepare_ldf_cache(m,branch)
        if lc is None or lc['ncl']<5:
            continue

        # equilibrium single/full tau on the same climate support, once
        pc=build_problem_cache(m,branch,PROCESS_MODE)
        gt=pc['climate_table'].set_index('climate_key').loc[lc['labels']].reset_index()
        kend_eq_single_full=float(
            kendalltau(gt.H_single,gt.H_full,variant='b').statistic
        )

        for tau in TAU_HOURS:
            for Ea in EA_KJMOL:
                Hf,Hfb,Nkf=run_ldf_cached(lc,tau,Ea,'full')
                Hs,Hsb,Nks=run_ldf_cached(lc,tau,Ea,'single')

                kend_eq_full=float(kendalltau(lc['H_eq_full'],Hf,variant='b').statistic)
                spear_eq_full=float(spearmanr(lc['H_eq_full'],Hf).statistic)
                kend_single_full=float(kendalltau(Hs,Hf,variant='b').statistic)
                spear_single_full=float(spearmanr(Hs,Hf).statistic)

                process_rows.append(dict(
                    material=m,branch=branch,tau_h=tau,Ea_kJmol=Ea,
                    n_climates=lc['ncl'],
                    kendall_Heq_vs_fullLDF=kend_eq_full,
                    spearman_Heq_vs_fullLDF=spear_eq_full,
                    kendall_singleLDF_vs_fullLDF=kend_single_full,
                    spearman_singleLDF_vs_fullLDF=spear_single_full,
                    kendall_eq_single_vs_full=kend_eq_single_full,
                    delta_kendall_dynamic_minus_equilibrium=(
                        kend_single_full-kend_eq_single_full
                    ),
                    median_kept_fraction=float(np.median(
                        Nkf/np.maximum(
                            np.bincount(lc['codes'],minlength=lc['ncl']),1
                        )
                    ))
                ))

                # compact climate-level output
                for ci,key in enumerate(lc['labels']):
                    process_detail_rows.append(dict(
                        material=m,branch=branch,tau_h=tau,Ea_kJmol=Ea,
                        climate_key=key,
                        H_eq_full=lc['H_eq_full'][ci],
                        H_LDF_full=Hf[ci],H_LDF_single=Hs[ci],
                        H_LDF_bal_full=Hfb[ci],H_LDF_bal_single=Hsb[ci],
                        N_kept_full=Nkf[ci],N_kept_single=Nks[ci],
                    ))

process_summary=pd.DataFrame(process_rows)
process_detail=pd.DataFrame(process_detail_rows)

process_summary.to_csv(OUT/'H_vs_LDF_rank_agreement_extended.csv',index=False)
process_detail.to_csv(OUT/'H_vs_LDF_climate_scores_extended.csv',index=False)

display(process_summary.sort_values(['material','branch','Ea_kJmol','tau_h']))

san=process_summary[
    np.isclose(process_summary.tau_h,1e-3)&
    np.isclose(process_summary.Ea_kJmol,0)
]
sanity_pass=bool(len(san)>0 and (san.kendall_Heq_vs_fullLDF>=0.999).all())
print('Sanity criterion tau_k=1e-3 h, Kendall >= 0.999:',
      'PASS' if sanity_pass else 'FAIL')
display(san)

fast=process_summary[
    (process_summary.tau_h<=1.0)&
    (process_summary.tau_h>=0.25)&
    np.isclose(process_summary.Ea_kJmol,0)
]
bridge_pass=bool(len(fast)>0 and (fast.kendall_Heq_vs_fullLDF>=0.9).all())
print('Bridge criterion tau_k<=1 h, constant-k Kendall >=0.9:',
      'PASS' if bridge_pass else 'FAIL')
display(fast)

if len(process_summary):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    for (m,b,Ea),g in process_summary.groupby(['material','branch','Ea_kJmol']):
        ax.plot(g.tau_h,g.kendall_Heq_vs_fullLDF,marker='o',
                label=f'{m}|{b}|Ea={Ea:g}')
    ax.set_xscale('log');ax.set_ylim(-1,1.02);ax.axhline(.9,linewidth=1)
    ax.set_xlabel('LDF time constant at Tref, τk (h)')
    ax.set_ylabel('Kendall τ: equilibrium H vs full-LDF')
    ax.set_title('Kinetic bridge for climate ordering')
    ax.legend(fontsize=6,ncol=2)
    fig.tight_layout()
    fig.savefig(FIG/'H_vs_fullLDF_rank_agreement.png',dpi=220)
    plt.show()

    fig,ax=plt.subplots(figsize=(7.2,4.6))
    for (m,b,Ea),g in process_summary.groupby(['material','branch','Ea_kJmol']):
        ax.plot(g.tau_h,g.delta_kendall_dynamic_minus_equilibrium,marker='o',
                label=f'{m}|{b}|Ea={Ea:g}')
    ax.set_xscale('log');ax.axhline(0,linewidth=1)
    ax.set_xlabel('LDF time constant at Tref, τk (h)')
    ax.set_ylabel('Δ Kendall τ = dynamic(single,full) − equilibrium(single,full)')
    ax.set_title('Does finite kinetics reduce or amplify temperature-information value?')
    ax.legend(fontsize=6,ncol=2)
    fig.tight_layout()
    fig.savefig(FIG/'single_full_LDF_information_shift.png',dpi=220)
    plt.show()



### Interpretation rule for Test D

- **Sanity:** \(\tau_k=10^{-3}\) h and \(E_a=0\) must give Kendall \(\tau\ge0.999\) between equilibrium \(H\) and full-LDF ranking.
- **Fast-kinetic bridge:** every constant-\(k\) case with \(0.25\le\tau_k\le1\) h should have Kendall \(\tau\ge0.9\).
- **Temperature-information attenuation:** positive
  \[
  \Delta\tau=
  \tau(\mathrm{single\!-\!LDF},\mathrm{full\!-\!LDF})
  -
  \tau(H_{\rm single},H_{\rm full})
  \]
  means finite kinetics makes the reduced representation look more similar to the fuller one.
- **Temperature-information amplification:** negative \(\Delta\tau\) means finite kinetics increases the ranking penalty of omitting temperature dependence.
- Arrhenius sensitivities determine whether the Mg low-temperature mechanism becomes weaker or stronger when kinetics slows at low \(T\).

These are bridge/sensitivity results, not productivity validation.


## 6. Submission-facing summary table

In [ ]:

summary_rows=[]

# A
if 'mg_interp_summary' in globals() and len(mg_interp_summary):
    summary_rows.append(dict(
        test='A. Mg interpolation sensitivity',
        preregistered_criterion='All 5×15 scheme-phase results have C<0',
        result=(
            f"{int(mg_interp_summary.n_C_negative.sum())}/"
            f"{int(mg_interp_summary.n_phases.sum())} C<0; "
            f"reversal counts={dict(zip(mg_interp_summary.scheme,mg_interp_summary.reversal_count.astype(int)))}"
        ),
        status='PASS' if (mg_interp_summary.frac_C_negative==1).all() else 'FAIL'
    ))

# B
if 'C_cell_summary' in globals() and len(C_cell_summary):
    h=int(C_cell_summary.held_interior.sum())
    summary_rows.append(dict(
        test='B. C generalization',
        preregistered_criterion='All classes included; primary scale |C|/Hbar; held-interior reported separately',
        result=(
            f"{len(C_cell_summary)} cells summarized; "
            f"{h} held-interior cells; "
            f"median cell |C|/Hbar={C_cell_summary.median_C_rel_Hbar.median():.4f}"
        ),
        status='DESCRIPTIVE'
    ))

# C
if 'family_results' in globals():
    nf=int(C_cell_summary.family_primary.nunique())
    summary_rows.append(dict(
        test='C. Family sensitivity',
        preregistered_criterion='Family results + LOFO + within-family climate bootstrap',
        result=f'{nf} primary families represented',
        status='LIMITED GENERALIZATION'
    ))

# D
if 'process_summary' in globals() and len(process_summary):
    summary_rows.append(dict(
        test='D. Kinetic bridge',
        preregistered_criterion='tau->0 sanity >=0.999; tau<=1h constant-k bridge >=0.9',
        result=(
            f"sanity={'PASS' if sanity_pass else 'FAIL'}; "
            f"fast bridge={'PASS' if bridge_pass else 'FAIL'}"
        ),
        status='PASS' if (sanity_pass and bridge_pass) else 'CHECK'
    ))

submission_summary=pd.DataFrame(summary_rows)
submission_summary.to_csv(OUT/'submission_gate_summary_T2.csv',index=False)
display(submission_summary)

print('\nSaved outputs to:',OUT)
print('Saved figures to:',FIG)


In [ ]:

print('Cached problem cells:', len(PROBLEM_CACHE))
print('Cached LDF problems:', len(LDF_CACHE))
display(timing_report())



## 7. Manuscript decision rules after T2

### A. Mg interpolation
The strongest result is not that monotone interpolation usually preserves the sign of a pointwise correction. Report:
- the **range of \(C\)** across plausible temperature warps;
- whether **rank reversals appear/disappear**;
- the discrepancy between the original `LinearND` fuller reference and direct endpoint-linear interpolation.

### B. Correction generalization
Use \(|C|/\bar H\) as the main magnitude scale. Keep \(|C|/|\Delta H_{\rm single}|\) only for the exact reversal condition.

The strongest route toward a prospective characterization rule is a repeated positive relationship between
\[
|C|/\bar H
\]
and differential occupancy-weighted temperature distance from \(T_{\rm ref}\) across independent families.

### C. Family uncertainty
With few independent families, report family-specific behavior and LOFO robustness. Interpret the hierarchical bootstrap strictly as **within-family climate uncertainty**, not population-level chemistry inference.

### D. Kinetic bridge
A fast-kinetic pass supports \(H\) as an equilibrium-opportunity ranking surrogate. The `single-LDF` versus `full-LDF` comparison determines whether kinetics reduces or increases the practical value of temperature-resolved equilibrium information.

A direct comparison to a published process-productivity ranking remains stronger evidence and should be added if compatible source data become available.


### Computational reproducibility
T3 changes only execution strategy. It does not change:
- support definitions;
- representation definitions;
- pre-registered pass/fail criteria;
- pair-level equations;
- family definitions;
- LDF equations or parameter grid.

For validation, T2 and T3 output CSVs can be diffed on a small run. Numerical differences beyond floating-point tolerance should be treated as an implementation bug.



# Reviewer-response extensions

The following sections address four points identified after Tests A–D:

1. **Full-reference interpolation dependence:** rerun the trajectory-null gate with a temperature-slice tensor reference rather than Q's 2D `LinearND` reference.
2. **Rank relevance of \(C\):** separate correction magnitude from the fraction of climate pairs for which the correction is large enough to threaten the single-isotherm ranking margin.
3. **Slow-kinetic sensitivity:** remove segment-wise spin-up censoring by reinitializing each supported segment at its first equilibrium state and scoring all supported transitions.
4. **Kinetic interpretation guardrail:** the LDF analysis remains a sensitivity bridge. Real amine-sorbent time scales are condition-dependent and may extend well beyond 1 h under cold/humid/high-PEI conditions.



## Test E — Replace `LinearND` full reference with a slice-wise tensor temperature interpolation

Q's `FullSurface` uses Delaunay triangulation in the \((RH,T)\) plane. If triangles cross RH knots, temperature interpolation can mix RH-direction structure. Test A showed that, for Mg lower, `LinearND` produced a systematically larger \(|C|\) than direct endpoint-linear temperature interpolation.

We therefore construct a second fuller reference:

1. linearly interpolate \(q(RH)\) within each measured temperature slice;
2. bracket each query temperature by adjacent measured temperature slices;
3. linearly interpolate only in temperature between the two RH-interpolated slice values.

Primary sensitivity target:
\[
\Delta\rho_{\rm tensor}
=
\rho_{S,F_{\rm tensor}\cdot N}
-
\rho_{RH,F_{\rm tensor}\cdot N}.
\]

Report class-C sign preservation, median \(\Delta\rho\) by branch, and agreement with the original `LinearND` gate. This tests reference construction sensitivity; neither reference is treated as truth.


In [ ]:

class TensorSliceFull:
    # Slice-wise RH interpolation + adjacent-temperature linear interpolation.
    def __init__(self,d):
        self.temps=np.array(sorted(d.temperature_C.unique()),float)
        self.fs={}
        for T in self.temps:
            z=d[d.temperature_C==T].sort_values('rh_pct')
            self.fs[float(T)]=interp1d(
                z.rh_pct.to_numpy(float),
                z.q_mmol_g.to_numpy(float),
                kind='linear',bounds_error=False,fill_value=np.nan
            )

    def eval(self,rh,T):
        rh=np.asarray(rh,float)
        T=np.asarray(T,float)
        out=np.full(np.broadcast(rh,T).shape,np.nan,float)

        for Tk in self.temps:
            mask=np.isclose(T,Tk,atol=1e-12)
            if np.any(mask):
                out[mask]=np.asarray(self.fs[float(Tk)](rh[mask]),float)

        for a,b in zip(self.temps[:-1],self.temps[1:]):
            mask=(T>a)&(T<b)
            if not np.any(mask):
                continue
            qa=np.asarray(self.fs[float(a)](rh[mask]),float)
            qb=np.asarray(self.fs[float(b)](rh[mask]),float)
            lam=(T[mask]-a)/(b-a)
            q=(1-lam)*qa+lam*qb
            q[~(np.isfinite(qa)&np.isfinite(qb))]=np.nan
            out[mask]=q
        return out

TENSOR_FULL={m:TensorSliceFull(materials[m]) for m in model}

def partial_spearman(x,y,z):
    x=np.asarray(x,float);y=np.asarray(y,float);z=np.asarray(z,float)
    ok=np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x=x[ok];y=y[ok];z=z[ok]
    if len(x)<5: return np.nan
    rxy=spearmanr(x,y).statistic
    rxz=spearmanr(x,z).statistic
    ryz=spearmanr(y,z).statistic
    den=np.sqrt(max(0,(1-rxz**2)*(1-ryz**2)))
    return np.nan if den<EPS else float((rxy-rxz*ryz)/den)

def tensor_problem_scores(material,branch,mode):
    c=build_problem_cache(material,branch,mode)
    if c is None:
        return pd.DataFrame()

    b=c['base'].copy()
    rh0=c['RH0'];rh1=c['RH1'];T0=c['T0'];T1=c['T1']
    qt0=TENSOR_FULL[material].eval(rh0,T0)
    qt1=TENSOR_FULL[material].eval(rh1,T1)
    ok=np.isfinite(qt0)&np.isfinite(qt1)
    if not np.any(ok):
        return pd.DataFrame()

    b=b.loc[ok].reset_index(drop=True)
    labels,codes=climate_codes_from_df(b)
    ncl=len(labels)

    Htensor=0.5*bincount_sum(codes,np.abs(qt1[ok]-qt0[ok]),ncl)
    Hsingle=0.5*bincount_sum(codes,np.abs(c['qs1'][ok]-c['qs0'][ok]),ncl)
    Hrh=0.5*bincount_sum(codes,np.abs(c['RH1'][ok]-c['RH0'][ok]),ncl)
    N=bincount_count(codes,ncl)

    return pd.DataFrame({
        'climate_key':labels,'H_tensor':Htensor,'H_single':Hsingle,'H_RH':Hrh,'N':N
    })

classC_cells=all_cells[
    all_cells.mechanism_class.astype(str).str.startswith('C_')
].copy()

tensor_gate_rows=[]
for _,r in classC_cells.iterrows():
    m,b,mode=r.material,r.branch,r.temporal_mode
    if m not in TENSOR_FULL or mode not in BASE_MODES:
        continue
    d=tensor_problem_scores(m,b,mode)
    if len(d)<5:
        continue
    rho_sf=partial_spearman(d.H_single,d.H_tensor,d.N)
    rho_rf=partial_spearman(d.H_RH,d.H_tensor,d.N)
    tensor_gate_rows.append(dict(
        material=m,branch=b,temporal_mode=mode,
        mechanism_class=r.mechanism_class,
        n_climates=len(d),
        rho_single_tensor_partial=rho_sf,
        rho_RH_tensor_partial=rho_rf,
        delta_rho_tensor=(rho_sf-rho_rf if np.isfinite(rho_sf) and np.isfinite(rho_rf) else np.nan),
    ))

tensor_gate=pd.DataFrame(tensor_gate_rows)

orig=gate.copy()
if 'delta_rho_partial' not in orig.columns and 'delta_rho_partial_single_minus_null' in orig.columns:
    orig['delta_rho_partial']=orig['delta_rho_partial_single_minus_null']

tensor_compare=tensor_gate.merge(
    orig[['material','branch','temporal_mode','delta_rho_partial']],
    on=['material','branch','temporal_mode'],how='left'
).rename(columns={'delta_rho_partial':'delta_rho_LinearND'})

tensor_compare['same_delta_sign']=(
    np.sign(tensor_compare.delta_rho_tensor)==
    np.sign(tensor_compare.delta_rho_LinearND)
)

tensor_gate.to_csv(OUT/'S2_classC_tensor_reference_gate.csv',index=False)
tensor_compare.to_csv(OUT/'S2_classC_tensor_vs_LinearND.csv',index=False)

print('Class-C tensor-reference summary')
display(
    tensor_gate.groupby('branch',as_index=False)
    .agg(
        n_cells=('delta_rho_tensor','size'),
        n_positive=('delta_rho_tensor',lambda x:int(np.sum(np.asarray(x)>0))),
        median_delta_rho=('delta_rho_tensor','median'),
        q25=('delta_rho_tensor',lambda x:np.nanquantile(x,.25)),
        q75=('delta_rho_tensor',lambda x:np.nanquantile(x,.75)),
    )
)

print('Overall sign agreement with LinearND:')
print(
    tensor_compare.same_delta_sign.mean(),
    f"({tensor_compare.same_delta_sign.sum()}/{len(tensor_compare)})"
)
display(
    tensor_compare.sort_values(
        ['same_delta_sign','material','branch','temporal_mode']
    )
)



### Test C extension — correction magnitude versus rank relevance

\(C_{\rm rel}=|C|/\bar H\) measures score-level correction size, not ranking relevance.

Report separately:
\[
f_{\rm margin}=P(|C|>|\Delta H_{\rm single}|)
\]
and the actual reversal fraction
\[
f_{\rm rev}=P[
\operatorname{sign}(C)\neq\operatorname{sign}(\Delta H_{\rm single}),
|C|>|\Delta H_{\rm single}|
].
\]

This is especially important for ImCOF and LWT: both are held-interior A-class cases, so temperature information is independently supported even though the RH slice itself is affine.


In [ ]:

if len(G_pairs):
    gp=G_pairs.copy()
    gp['family_primary']=gp.material.map(chemistry_family_primary)
    gp['margin_threat']=np.abs(gp.C)>np.abs(gp.delta_Hsingle)

    family_rank_relevance=(
        gp.groupby('family_primary',as_index=False)
        .agg(
            n_pairs=('reversal','size'),
            median_C_rel_Hbar=('C_rel_Hbar','median'),
            frac_margin_threat=('margin_threat','mean'),
            reversal_fraction=('reversal','mean'),
        )
    )
    family_rank_relevance.to_csv(
        OUT/'family_C_magnitude_vs_rank_relevance.csv',index=False
    )
    display(family_rank_relevance)

    held=gp[gp.held_interior].copy()
    held_rank=(
        held.groupby(['material','branch'],as_index=False)
        .agg(
            n_pairs=('reversal','size'),
            median_C_rel_Hbar=('C_rel_Hbar','median'),
            frac_margin_threat=('margin_threat','mean'),
            reversal_fraction=('reversal','mean'),
        )
    )
    held_rank.to_csv(
        OUT/'held_interior_C_rank_relevance.csv',index=False
    )
    print('Held-interior independent temperature-information cases')
    display(held_rank)



## Test D extension — slow-kinetic sensitivity without segment-wise spin-up censoring

The primary Test D remains the matched-support analysis for \(\tau_k\le1\) h.

For \(\tau_k\ge3\) h, removing \(5\tau_k\) after every supported-segment restart destroys the retained cohort. As a sensitivity analysis, each supported segment is initialized at
\[
q(t_0)=q^*(t_0)
\]
and **all supported transitions are scored**.

This eliminates artificial support thinning, but does not propagate the state through unsupported gaps. It is therefore a segment-reinitialized slow-kinetic sensitivity, not a fully continuous process simulation.


In [ ]:

@njit(cache=True)
def _ldf_run_no_spinup(target,init_q,T1,codes,starts,ends,ncl,
                       tau_h,Ea_kJmol,Tref_C,dt,Rgas):
    H=np.zeros(ncl,np.float64)
    Hbal=np.zeros(ncl,np.float64)
    N=np.zeros(ncl,np.int64)

    kref=1.0/max(tau_h,1e-12)
    TrefK=Tref_C+273.15
    Ea=Ea_kJmol*1000.0

    for s in range(len(starts)):
        a=starts[s]; b=ends[s]
        if b<=a:
            continue

        q=init_q[a]
        pos=0.0;neg=0.0
        cc=codes[a]

        for p in range(a,b):
            if Ea_kJmol==0.0:
                kT=kref
            else:
                TK=T1[p]+273.15
                kT=kref*np.exp(-(Ea/Rgas)*(1.0/TK-1.0/TrefK))
            alpha=1.0-np.exp(-kT*dt)
            qn=q+alpha*(target[p]-q)
            dq=qn-q

            H[cc]+=0.5*abs(dq)
            if dq>=0:
                pos+=dq
            else:
                neg+=-dq
            N[cc]+=1
            q=qn

        Hbal[cc]+=min(pos,neg)

    return H,Hbal,N

def run_ldf_no_spinup(cache,tau_h,Ea_kJmol,which='full'):
    target=cache['qf_target'] if which=='full' else cache['qs_target']
    init_q=cache['qf_init'] if which=='full' else cache['qs_init']
    return _ldf_run_no_spinup(
        np.asarray(target,float),np.asarray(init_q,float),
        np.asarray(cache['T1'],float),np.asarray(cache['codes'],np.int64),
        np.asarray(cache['starts'],np.int64),np.asarray(cache['ends'],np.int64),
        int(cache['ncl']),float(tau_h),float(Ea_kJmol),
        float(cache['Tref']),float(cache['dt']),float(R_GAS)
    )

slow_rows=[]
for m in PROCESS_MATERIALS:
    for branch in PROCESS_BRANCHES:
        lc=prepare_ldf_cache(m,branch)
        if lc is None or lc['ncl']<5:
            continue

        pc=build_problem_cache(m,branch,PROCESS_MODE)
        gt=pc['climate_table'].set_index('climate_key').loc[lc['labels']].reset_index()
        eq_tau=float(kendalltau(gt.H_single,gt.H_full,variant='b').statistic)

        for tau in [3,6,12]:
            for Ea in EA_KJMOL:
                Hf,_,Nf=run_ldf_no_spinup(lc,tau,Ea,'full')
                Hs,_,Ns=run_ldf_no_spinup(lc,tau,Ea,'single')
                valid=(Nf>0)&(Ns>0)
                if valid.sum()<5:
                    continue

                k_eq_dyn=float(
                    kendalltau(
                        gt.H_full.to_numpy()[valid],Hf[valid],variant='b'
                    ).statistic
                )
                k_sf_dyn=float(
                    kendalltau(Hs[valid],Hf[valid],variant='b').statistic
                )

                slow_rows.append(dict(
                    material=m,branch=branch,tau_h=tau,Ea_kJmol=Ea,
                    n_climates=int(valid.sum()),
                    kendall_equilibriumH_vs_fullLDF=k_eq_dyn,
                    kendall_singleLDF_vs_fullLDF=k_sf_dyn,
                    kendall_eq_single_vs_full=eq_tau,
                    delta_dynamic_minus_equilibrium=k_sf_dyn-eq_tau,
                    retained_fraction=1.0,
                ))

slow_sensitivity=pd.DataFrame(slow_rows)
slow_sensitivity.to_csv(
    OUT/'slow_LDF_segment_reinitialized_sensitivity.csv',index=False
)
display(
    slow_sensitivity.sort_values(
        ['material','branch','Ea_kJmol','tau_h']
    )
)

if len(slow_sensitivity):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    for (m,b,Ea),g in slow_sensitivity.groupby(
        ['material','branch','Ea_kJmol']
    ):
        ax.plot(
            g.tau_h,g.kendall_equilibriumH_vs_fullLDF,
            marker='o',label=f'{m}|{b}|Ea={Ea:g}'
        )
    ax.axhline(.9,linewidth=1)
    ax.set_xlabel('LDF time constant at Tref, tau_k (h)')
    ax.set_ylabel('Kendall tau: equilibrium H vs full-LDF')
    ax.set_title('Slow-kinetic sensitivity without spin-up censoring')
    ax.legend(fontsize=6,ncol=2)
    fig.tight_layout()
    fig.savefig(FIG/'slow_LDF_no_spinup_sensitivity.png',dpi=220)
    plt.show()



## Kinetic interpretation guardrail

The LDF sweep should not be presented as if \(\tau_k\le1\) h were universally realistic for amine DAC sorbents.

Recent PEI/ePTFE/silica DAC measurements show strongly condition-dependent kinetics. Under cold and humid conditions, adsorption can remain unequilibrated for many hours, especially at high PEI loading. Therefore:

- \(\tau_k\le1\) h is a **fast-kinetic regime**, not a universal material property;
- the bridge result asks whether equilibrium climate ordering survives when response is fast relative to hourly weather;
- slow-kinetic sensitivity is directly relevant to cold/humid diffusion-limited aminopolymer conditions;
- material-specific kinetic calibration is required before interpreting \(\tau_k\) quantitatively.



# T5 reviewer-response refinements

This section supersedes the earlier Test E / Test C extension / slow-LDF extension where noted.

## Pre-registered criteria

### Tensor-reference gate
The main S2 conclusion is considered preserved if:

1. at least **90% of class-C cells** retain the sign of \(\Delta\rho\) when `LinearND` is replaced by the tensor reference on a matched support;
2. the median \(\Delta\rho\) remains positive in both lower and upper branches.

### Independent held-interior nested result
The temperature-information conclusion is considered qualitatively preserved if, under the tensor reference:

- upper-branch median \(\Delta\tau\) across ImCOF-TAEA and LWT-bPPI5 remains positive;
- lower branch does not acquire a comparably consistent positive gain.

### Tensor implementation unit test
For Mg lower, the tensor reference must reproduce Test A `endpoint_linear` \(C\) values phase-by-phase to numerical tolerance:
\[
\max |C_{\rm tensor}-C_{\rm endpoint-linear}| < 10^{-10}.
\]

### Rank relevance of \(C\)
Report separately:
- score-level magnitude \(C_{\rm rel}=|C|/\bar H\);
- margin-threat fraction \(P(|C|>|\Delta H_{\rm single}|)\);
- reversal fraction;
- conditional reversal probability among margin-threatened pairs.

All rank-relevance summaries are also repeated after excluding relative single-isotherm margins below 5%.

### Slow-LDF sensitivity
For \(\tau_k\ge3\) h, report:
- supported-segment length distribution relative to \(\tau_k\);
- segment-reinitialized results under two initializations:
  1. \(q(t_0)=q^*(t_0)\);
  2. \(q(t_0)=\overline{q^*}_{\rm segment}\).

This is explicitly a **segment-reinitialized slow-kinetic sensitivity**, not a continuous process simulation.



## Test E refined — separate reference sensitivity from support sensitivity

Two analyses are reported.

### E1. Matched-support reference swap
Start from the original Q support, then restrict to transitions for which both `LinearND` and tensor references are finite. On that exact transition set, score:

- `LinearND` full;
- tensor full;
- single reference isotherm;
- RH-trajectory null.

Thus the climate cohort and transition set are identical when comparing the two fuller references.

### E2. Native tensor support
Rebuild support using the tensor representation itself. This shows the combined effect of changing both the reference construction and its admissible RH–T support.

A/B/C mechanism labels are **not recomputed**, because those labels depend only on the single isotherm and RH-trajectory null.


In [ ]:

class TensorSliceFull:
    def __init__(self,d):
        self.temps=np.array(sorted(d.temperature_C.unique()),float)
        self.fs={}
        self.ranges={}
        for T in self.temps:
            z=d[d.temperature_C==T].sort_values('rh_pct')
            self.fs[float(T)]=interp1d(
                z.rh_pct.to_numpy(float),
                z.q_mmol_g.to_numpy(float),
                kind='linear',bounds_error=False,fill_value=np.nan
            )
            self.ranges[float(T)]=(float(z.rh_pct.min()),float(z.rh_pct.max()))

    def eval(self,rh,T):
        rh=np.asarray(rh,float); T=np.asarray(T,float)
        out=np.full(np.broadcast(rh,T).shape,np.nan,float)

        for Tk in self.temps:
            mask=np.isclose(T,Tk,atol=1e-12)
            if np.any(mask):
                out[mask]=np.asarray(self.fs[float(Tk)](rh[mask]),float)

        for a,b in zip(self.temps[:-1],self.temps[1:]):
            mask=(T>a)&(T<b)
            if not np.any(mask):
                continue
            qa=np.asarray(self.fs[float(a)](rh[mask]),float)
            qb=np.asarray(self.fs[float(b)](rh[mask]),float)
            lam=(T[mask]-a)/(b-a)
            q=(1-lam)*qa+lam*qb
            q[~(np.isfinite(qa)&np.isfinite(qb))]=np.nan
            out[mask]=q
        return out

TENSOR_FULL={m:TensorSliceFull(materials[m]) for m in model}

def _partial_spearman(x,y,z):
    x=np.asarray(x,float); y=np.asarray(y,float); z=np.asarray(z,float)
    ok=np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x=x[ok]; y=y[ok]; z=z[ok]
    if len(x)<5: return np.nan
    rxy=spearmanr(x,y).statistic
    rxz=spearmanr(x,z).statistic
    ryz=spearmanr(y,z).statistic
    den=np.sqrt(max(0.0,(1-rxz**2)*(1-ryz**2)))
    return np.nan if den<EPS else float((rxy-rxz*ryz)/den)

def _aggregate_scores(labels,codes,Hlin,Hten,Hsingle,Hrh):
    ncl=len(labels)
    N=bincount_count(codes,ncl)
    return pd.DataFrame({
        'climate_key':labels,
        'H_linearND':Hlin,
        'H_tensor':Hten,
        'H_single':Hsingle,
        'H_RH':Hrh,
        'N':N
    })

def matched_support_reference_scores(material,branch,mode):
    c=build_problem_cache(material,branch,mode)
    if c is None: return pd.DataFrame()

    rh0,rh1,T0,T1=c['RH0'],c['RH1'],c['T0'],c['T1']
    qt0=TENSOR_FULL[material].eval(rh0,T0)
    qt1=TENSOR_FULL[material].eval(rh1,T1)
    ok=np.isfinite(qt0)&np.isfinite(qt1)&np.isfinite(c['qf0'])&np.isfinite(c['qf1'])
    if not np.any(ok): return pd.DataFrame()

    b=c['base'].loc[ok].reset_index(drop=True)
    labels,codes=climate_codes_from_df(b)
    ncl=len(labels)

    Hlin=0.5*bincount_sum(codes,np.abs(c['qf1'][ok]-c['qf0'][ok]),ncl)
    Hten=0.5*bincount_sum(codes,np.abs(qt1[ok]-qt0[ok]),ncl)
    Hsingle=0.5*bincount_sum(codes,np.abs(c['qs1'][ok]-c['qs0'][ok]),ncl)
    Hrh=0.5*bincount_sum(codes,np.abs(rh1[ok]-rh0[ok]),ncl)
    return _aggregate_scores(labels,codes,Hlin,Hten,Hsingle,Hrh)

def tensor_native_base(material,branch,mode):
    er=elig.set_index('material').loc[material]
    Tlo,Thi=branch_bounds(er,branch)
    s=model[material]['single_ref']
    Rlo,Rhi=float(s.rmin),float(s.rmax)

    b=base_transitions(BASE_MODES[mode],Tlo,Thi,Rlo,Rhi)
    if b.empty: return b

    rh0=b.RH0.to_numpy(float); rh1=b.RH1.to_numpy(float)
    T0=b.T0.to_numpy(float); T1=b.T1.to_numpy(float)

    qt0=TENSOR_FULL[material].eval(rh0,T0)
    qt1=TENSOR_FULL[material].eval(rh1,T1)
    qs0=s.eval(rh0,T0); qs1=s.eval(rh1,T1)

    ok=np.isfinite(qt0)&np.isfinite(qt1)&np.isfinite(qs0)&np.isfinite(qs1)
    return b.loc[ok].reset_index(drop=True)

def tensor_native_scores(material,branch,mode):
    b=tensor_native_base(material,branch,mode)
    if b.empty: return pd.DataFrame()

    rh0=b.RH0.to_numpy(float); rh1=b.RH1.to_numpy(float)
    T0=b.T0.to_numpy(float); T1=b.T1.to_numpy(float)

    qt0=TENSOR_FULL[material].eval(rh0,T0)
    qt1=TENSOR_FULL[material].eval(rh1,T1)
    qs0=model[material]['single_ref'].eval(rh0,T0)
    qs1=model[material]['single_ref'].eval(rh1,T1)

    labels,codes=climate_codes_from_df(b)
    ncl=len(labels)
    Hten=0.5*bincount_sum(codes,np.abs(qt1-qt0),ncl)
    Hsingle=0.5*bincount_sum(codes,np.abs(qs1-qs0),ncl)
    Hrh=0.5*bincount_sum(codes,np.abs(rh1-rh0),ncl)
    N=bincount_count(codes,ncl)

    return pd.DataFrame({
        'climate_key':labels,'H_tensor':Hten,'H_single':Hsingle,'H_RH':Hrh,'N':N
    })

classC_cells=all_cells[
    all_cells.mechanism_class.astype(str).str.startswith('C_')
].copy()

rows=[]
for _,r in classC_cells.iterrows():
    m,b,mode=r.material,r.branch,r.temporal_mode
    if m not in TENSOR_FULL or mode not in BASE_MODES:
        continue

    dm=matched_support_reference_scores(m,b,mode)
    if len(dm)>=5:
        rho_s_lin=_partial_spearman(dm.H_single,dm.H_linearND,dm.N)
        rho_r_lin=_partial_spearman(dm.H_RH,dm.H_linearND,dm.N)
        rho_s_ten=_partial_spearman(dm.H_single,dm.H_tensor,dm.N)
        rho_r_ten=_partial_spearman(dm.H_RH,dm.H_tensor,dm.N)
        rows.append(dict(
            support_mode='matched_Q_tensor',
            material=m,branch=b,temporal_mode=mode,
            mechanism_class=r.mechanism_class,
            n_climates=len(dm),
            delta_rho_LinearND=rho_s_lin-rho_r_lin,
            delta_rho_tensor=rho_s_ten-rho_r_ten,
        ))

    dn=tensor_native_scores(m,b,mode)
    if len(dn)>=5:
        rho_s_ten=_partial_spearman(dn.H_single,dn.H_tensor,dn.N)
        rho_r_ten=_partial_spearman(dn.H_RH,dn.H_tensor,dn.N)
        rows.append(dict(
            support_mode='tensor_native',
            material=m,branch=b,temporal_mode=mode,
            mechanism_class=r.mechanism_class,
            n_climates=len(dn),
            delta_rho_LinearND=np.nan,
            delta_rho_tensor=rho_s_ten-rho_r_ten,
        ))

tensor_ref_sensitivity=pd.DataFrame(rows)
tensor_ref_sensitivity['same_sign_vs_LinearND']=np.where(
    tensor_ref_sensitivity.support_mode.eq('matched_Q_tensor'),
    np.sign(tensor_ref_sensitivity.delta_rho_tensor)==
    np.sign(tensor_ref_sensitivity.delta_rho_LinearND),
    np.nan
)
tensor_ref_sensitivity.to_csv(
    OUT/'T5_tensor_reference_support_sensitivity.csv',index=False
)

e1=tensor_ref_sensitivity[
    tensor_ref_sensitivity.support_mode=='matched_Q_tensor'
].copy()

print('E1 matched-support summary')
display(
    e1.groupby('branch',as_index=False)
      .agg(
          n_cells=('delta_rho_tensor','size'),
          sign_agreement=('same_sign_vs_LinearND','mean'),
          median_delta_LinearND=('delta_rho_LinearND','median'),
          median_delta_tensor=('delta_rho_tensor','median'),
          n_tensor_positive=('delta_rho_tensor',lambda x:int(np.sum(np.asarray(x)>0))),
      )
)

overall_sign=float(e1.same_sign_vs_LinearND.mean()) if len(e1) else np.nan
med_lower=float(e1.loc[e1.branch=='lower','delta_rho_tensor'].median()) if np.any(e1.branch=='lower') else np.nan
med_upper=float(e1.loc[e1.branch=='upper','delta_rho_tensor'].median()) if np.any(e1.branch=='upper') else np.nan

gate_E_pass=(
    np.isfinite(overall_sign) and overall_sign>=0.90 and
    np.isfinite(med_lower) and med_lower>0 and
    np.isfinite(med_upper) and med_upper>0
)

print('Pre-registered tensor-reference gate:',
      'PASS' if gate_E_pass else 'FAIL')
print('overall sign agreement =',overall_sign,
      '| median lower =',med_lower,
      '| median upper =',med_upper)

print('\nE2 tensor-native support summary')
e2=tensor_ref_sensitivity[
    tensor_ref_sensitivity.support_mode=='tensor_native'
]
display(
    e2.groupby('branch',as_index=False)
      .agg(
          n_cells=('delta_rho_tensor','size'),
          median_delta_tensor=('delta_rho_tensor','median'),
          q25=('delta_rho_tensor',lambda x:np.nanquantile(x,.25)),
          q75=('delta_rho_tensor',lambda x:np.nanquantile(x,.75)),
          n_tensor_positive=('delta_rho_tensor',lambda x:int(np.sum(np.asarray(x)>0))),
      )
)



### E3. Unit test and independent held-interior nested sensitivity

The Mg lower branch is a direct implementation check: within \([T_{\rm low},T_{\rm ref}]\), the tensor reference should be identical to Test A's endpoint-linear interpolation.

For ImCOF-TAEA and LWT-bPPI5, recompute nested \(\Delta\tau\) against the tensor fuller reference. These two materials are the only held-interior cases and therefore the direct evidence for independently measured temperature information.


In [ ]:

unit_rows=[]
for (family,off),pc in MG_PHASE_CACHE.items():
    qt0=TENSOR_FULL[MG].eval(pc['rh0'],pc['T0'])
    qt1=TENSOR_FULL[MG].eval(pc['rh1'],pc['T1'])
    Hten=0.5*bincount_sum(pc['codes'],np.abs(qt1-qt0),pc['ncl'])
    d_ten=pair_delta_fast(pc['labels'],Hten)
    d_single=pair_delta_fast(pc['labels'],pc['Hs'])
    C_ten=d_ten-d_single

    refrow=mg_interp[
        (mg_interp.phase_family==family)&
        (mg_interp.offset_h==off)&
        (mg_interp.scheme=='endpoint_linear')
    ]
    C_ep=float(refrow.C.iloc[0]) if len(refrow)==1 else np.nan

    unit_rows.append(dict(
        phase_family=family,offset_h=off,
        C_tensor=C_ten,C_endpoint_linear=C_ep,
        abs_diff=abs(C_ten-C_ep)
    ))

tensor_unit=pd.DataFrame(unit_rows)
tensor_unit.to_csv(OUT/'T5_Mg_tensor_endpointlinear_unit_test.csv',index=False)
unit_max=float(tensor_unit.abs_diff.max())
print('Mg lower tensor vs endpoint-linear max |ΔC| =',unit_max)
print('UNIT TEST:', 'PASS' if unit_max<1e-10 else 'FAIL')
display(tensor_unit)

HELD_MATERIALS=[m for m in ['ImCOF-TAEA','LWT-bPPI5'] if m in model]

def nested_rep_name(branch):
    return 'nested_low' if branch=='lower' else 'nested_high'

def held_nested_tensor(material,branch,mode,support_mode='matched'):
    if support_mode=='matched':
        c=build_problem_cache(material,branch,mode)
        if c is None: return None
        b=c['base'].copy()

        rh0,rh1,T0,T1=c['RH0'],c['RH1'],c['T0'],c['T1']
        qt0=TENSOR_FULL[material].eval(rh0,T0)
        qt1=TENSOR_FULL[material].eval(rh1,T1)
        qn0=model[material][nested_rep_name(branch)].eval(rh0,T0)
        qn1=model[material][nested_rep_name(branch)].eval(rh1,T1)
        qs0,qs1=c['qs0'],c['qs1']

        ok=(np.isfinite(qt0)&np.isfinite(qt1)&
            np.isfinite(qn0)&np.isfinite(qn1)&
            np.isfinite(qs0)&np.isfinite(qs1))
        b=b.loc[ok].reset_index(drop=True)
        qt0,qt1,qn0,qn1,qs0,qs1=[
            x[ok] for x in [qt0,qt1,qn0,qn1,qs0,qs1]
        ]
    else:
        b=tensor_native_base(material,branch,mode)
        if b.empty: return None
        rh0=b.RH0.to_numpy(float); rh1=b.RH1.to_numpy(float)
        T0=b.T0.to_numpy(float); T1=b.T1.to_numpy(float)

        qt0=TENSOR_FULL[material].eval(rh0,T0)
        qt1=TENSOR_FULL[material].eval(rh1,T1)
        qn0=model[material][nested_rep_name(branch)].eval(rh0,T0)
        qn1=model[material][nested_rep_name(branch)].eval(rh1,T1)
        qs0=model[material]['single_ref'].eval(rh0,T0)
        qs1=model[material]['single_ref'].eval(rh1,T1)

        ok=(np.isfinite(qt0)&np.isfinite(qt1)&
            np.isfinite(qn0)&np.isfinite(qn1)&
            np.isfinite(qs0)&np.isfinite(qs1))
        b=b.loc[ok].reset_index(drop=True)
        qt0,qt1,qn0,qn1,qs0,qs1=[
            x[ok] for x in [qt0,qt1,qn0,qn1,qs0,qs1]
        ]

    if len(b)==0: return None

    labels,codes=climate_codes_from_df(b)
    ncl=len(labels)
    Ht=0.5*bincount_sum(codes,np.abs(qt1-qt0),ncl)
    Hs=0.5*bincount_sum(codes,np.abs(qs1-qs0),ncl)
    Hn=0.5*bincount_sum(codes,np.abs(qn1-qn0),ncl)

    if ncl<5: return None

    tau_s=float(kendalltau(Hs,Ht,variant='b').statistic)
    tau_n=float(kendalltau(Hn,Ht,variant='b').statistic)

    return dict(
        n_climates=ncl,
        tau_single=tau_s,
        tau_nested=tau_n,
        delta_tau=tau_n-tau_s
    )

held_rows=[]
for m in HELD_MATERIALS:
    for branch in ['lower','upper']:
        for mode in BASE_MODES:
            for smode in ['matched','tensor_native']:
                res=held_nested_tensor(m,branch,mode,smode)
                if res:
                    held_rows.append(dict(
                        material=m,branch=branch,temporal_mode=mode,
                        support_mode=smode,**res
                    ))

held_tensor_nested=pd.DataFrame(held_rows)
held_tensor_nested.to_csv(
    OUT/'T5_held_interior_nested_tensor_reference.csv',index=False
)
display(held_tensor_nested)

print('Held-interior qualitative summary')
display(
    held_tensor_nested.groupby(['support_mode','branch'],as_index=False)
    .agg(
        n_cells=('delta_tau','size'),
        median_delta_tau=('delta_tau','median'),
        n_positive=('delta_tau',lambda x:int(np.sum(np.asarray(x)>0))),
    )
)



## Test C refined — magnitude, margin threat, and actual reversal

A large \(C_{\rm rel}\) is a score-level effect. Ranking relevance is reported separately.

For each family:

\[
f_{\rm margin}=P(|C|>|\Delta H_{\rm single}|)
\]

\[
f_{\rm rev}=P(\mathrm{reversal})
\]

\[
P(\mathrm{reversal}\mid\mathrm{margin\ threat})
=
\frac{f_{\rm rev}}{f_{\rm margin}}
\]

when \(f_{\rm margin}>0\).

Because tiny single-isotherm pair margins inflate \(f_{\rm margin}\), all quantities are reported both:
- unfiltered;
- after excluding pairs with relative single-isotherm margin \(<5\%\).


In [ ]:

def rank_relevance_summary(df,margin_cut=None):
    z=df.copy()
    if margin_cut is not None:
        z=z[z.single_relative_margin>=margin_cut].copy()

    z['margin_threat']=np.abs(z.C)>np.abs(z.delta_Hsingle)

    out=[]
    for fam,g in z.groupby('family_primary'):
        n=len(g)
        fm=float(g.margin_threat.mean()) if n else np.nan
        fr=float(g.reversal.mean()) if n else np.nan
        out.append(dict(
            family_primary=fam,
            margin_filter=(0.0 if margin_cut is None else float(margin_cut)),
            n_pairs=n,
            median_C_rel_Hbar=float(g.C_rel_Hbar.median()) if n else np.nan,
            frac_margin_threat=fm,
            reversal_fraction=fr,
            conditional_reversal_given_threat=(fr/fm if fm>0 else np.nan)
        ))
    return pd.DataFrame(out)

gp=G_pairs.copy()
gp['family_primary']=gp.material.map(chemistry_family_primary)

rank_raw=rank_relevance_summary(gp,None)
rank_5=rank_relevance_summary(gp,0.05)
rank_family=pd.concat([rank_raw,rank_5],ignore_index=True)

rank_family.to_csv(
    OUT/'T5_family_rank_relevance_margin_filtered.csv',index=False
)
display(rank_family)

held=gp[gp.held_interior].copy()
held_rows=[]

for cut in [None,0.05]:
    z=held if cut is None else held[held.single_relative_margin>=cut]

    for (m,b),g in z.groupby(['material','branch']):
        fm=float((np.abs(g.C)>np.abs(g.delta_Hsingle)).mean())
        fr=float(g.reversal.mean())

        held_rows.append(dict(
            material=m,branch=b,
            margin_filter=(0.0 if cut is None else cut),
            n_pairs=len(g),
            median_C_rel_Hbar=float(g.C_rel_Hbar.median()),
            frac_margin_threat=fm,
            reversal_fraction=fr,
            conditional_reversal_given_threat=(fr/fm if fm>0 else np.nan)
        ))

held_rank_relevance=pd.DataFrame(held_rows)
held_rank_relevance.to_csv(
    OUT/'T5_held_interior_rank_relevance_margin_filtered.csv',index=False
)
display(held_rank_relevance)



## Test D refined — segment length and initialization sensitivity

For each material × branch, the supported-segment length distribution is compared with \(\tau_k\).

For \(\tau_k=3,6,12\) h report:
- median segment length;
- fraction of supported segments shorter than \(\tau_k\);
- fraction shorter than \(3\tau_k\);
- fraction shorter than \(5\tau_k\).

Then repeat the segment-reinitialized LDF with two initial states:

1. `first_eq`: \(q(t_0)=q^*(t_0)\);
2. `segment_mean`: \(q(t_0)=\overline{q^*}_{\rm segment}\).

If climate-ranking conclusions change materially between these initializations, the slow-\(\tau_k\) result is initialization-sensitive and should remain SI-only.


In [ ]:

seg_rows=[]
for m in PROCESS_MATERIALS:
    for branch in PROCESS_BRANCHES:
        lc=prepare_ldf_cache(m,branch)
        if lc is None: continue

        lengths=(lc['ends']-lc['starts']).astype(float)*float(lc['dt'])

        for tau in [3.0,6.0,12.0]:
            seg_rows.append(dict(
                material=m,branch=branch,tau_h=tau,
                n_segments=len(lengths),
                segment_median_h=float(np.median(lengths)),
                segment_q25_h=float(np.quantile(lengths,.25)),
                segment_q75_h=float(np.quantile(lengths,.75)),
                frac_len_lt_tau=float(np.mean(lengths<tau)),
                frac_len_lt_3tau=float(np.mean(lengths<3*tau)),
                frac_len_lt_5tau=float(np.mean(lengths<5*tau)),
            ))

segment_length_summary=pd.DataFrame(seg_rows)
segment_length_summary.to_csv(
    OUT/'T5_slow_LDF_segment_length_vs_tau.csv',index=False
)
display(segment_length_summary)

@njit(cache=True)
def _ldf_segment_reinit(target,init_q,T1,codes,starts,ends,ncl,
                        tau_h,Ea_kJmol,Tref_C,dt,Rgas,init_mode):
    H=np.zeros(ncl,np.float64)
    Hbal=np.zeros(ncl,np.float64)
    N=np.zeros(ncl,np.int64)

    kref=1.0/max(tau_h,1e-12)
    TrefK=Tref_C+273.15
    Ea=Ea_kJmol*1000.0

    for s in range(len(starts)):
        a=starts[s]; b=ends[s]
        if b<=a: continue

        if init_mode==0:
            q=init_q[a]
        else:
            acc=0.0
            for p in range(a,b):
                acc+=target[p]
            q=acc/max(1,b-a)

        pos=0.0; neg=0.0
        cc=codes[a]

        for p in range(a,b):
            if Ea_kJmol==0.0:
                kT=kref
            else:
                TK=T1[p]+273.15
                kT=kref*np.exp(-(Ea/Rgas)*(1.0/TK-1.0/TrefK))

            alpha=1.0-np.exp(-kT*dt)
            qn=q+alpha*(target[p]-q)
            dq=qn-q

            H[cc]+=0.5*abs(dq)
            if dq>=0:
                pos+=dq
            else:
                neg+=-dq

            N[cc]+=1
            q=qn

        Hbal[cc]+=min(pos,neg)

    return H,Hbal,N

def run_segment_reinit(cache,tau_h,Ea_kJmol,which='full',init_scheme='first_eq'):
    target=cache['qf_target'] if which=='full' else cache['qs_target']
    init_q=cache['qf_init'] if which=='full' else cache['qs_init']
    init_mode=0 if init_scheme=='first_eq' else 1

    return _ldf_segment_reinit(
        np.asarray(target,float),np.asarray(init_q,float),
        np.asarray(cache['T1'],float),np.asarray(cache['codes'],np.int64),
        np.asarray(cache['starts'],np.int64),np.asarray(cache['ends'],np.int64),
        int(cache['ncl']),float(tau_h),float(Ea_kJmol),
        float(cache['Tref']),float(cache['dt']),float(R_GAS),int(init_mode)
    )

slow_rows=[]

for m in PROCESS_MATERIALS:
    for branch in PROCESS_BRANCHES:
        lc=prepare_ldf_cache(m,branch)
        if lc is None or lc['ncl']<5: continue

        pc=build_problem_cache(m,branch,PROCESS_MODE)
        gt=pc['climate_table'].set_index('climate_key').loc[lc['labels']].reset_index()
        eq_tau=float(kendalltau(gt.H_single,gt.H_full,variant='b').statistic)

        for init_scheme in ['first_eq','segment_mean']:
            for tau in [3.0,6.0,12.0]:
                for Ea in EA_KJMOL:
                    Hf,_,Nf=run_segment_reinit(lc,tau,Ea,'full',init_scheme)
                    Hs,_,Ns=run_segment_reinit(lc,tau,Ea,'single',init_scheme)

                    valid=(Nf>0)&(Ns>0)
                    if valid.sum()<5: continue

                    k_eq_dyn=float(
                        kendalltau(
                            gt.H_full.to_numpy()[valid],Hf[valid],variant='b'
                        ).statistic
                    )
                    k_sf_dyn=float(
                        kendalltau(
                            Hs[valid],Hf[valid],variant='b'
                        ).statistic
                    )

                    slow_rows.append(dict(
                        material=m,branch=branch,init_scheme=init_scheme,
                        tau_h=tau,Ea_kJmol=Ea,n_climates=int(valid.sum()),
                        kendall_equilibriumH_vs_fullLDF=k_eq_dyn,
                        kendall_singleLDF_vs_fullLDF=k_sf_dyn,
                        kendall_eq_single_vs_full=eq_tau,
                        delta_dynamic_minus_equilibrium=k_sf_dyn-eq_tau
                    ))

slow_init_sensitivity=pd.DataFrame(slow_rows)
slow_init_sensitivity.to_csv(
    OUT/'T5_slow_LDF_initialization_sensitivity.csv',index=False
)
display(
    slow_init_sensitivity.sort_values(
        ['material','branch','Ea_kJmol','tau_h','init_scheme']
    )
)

wide=slow_init_sensitivity.pivot_table(
    index=['material','branch','tau_h','Ea_kJmol'],
    columns='init_scheme',
    values=[
        'kendall_equilibriumH_vs_fullLDF',
        'kendall_singleLDF_vs_fullLDF',
        'delta_dynamic_minus_equilibrium'
    ]
)

wide.columns=['__'.join(c) for c in wide.columns]
wide=wide.reset_index()

for metric in [
    'kendall_equilibriumH_vs_fullLDF',
    'kendall_singleLDF_vs_fullLDF',
    'delta_dynamic_minus_equilibrium'
]:
    a=f'{metric}__first_eq'
    b=f'{metric}__segment_mean'
    if a in wide.columns and b in wide.columns:
        wide[f'absdiff__{metric}']=np.abs(wide[a]-wide[b])

wide.to_csv(
    OUT/'T5_slow_LDF_initialization_difference.csv',index=False
)
display(wide)
